In [ ]:
# ablate_runner.py
# ============================================================
# [이 코드가 뭐 하는 코드인지 쉽게 설명]
#
# 이 코드는 "부분적으로만 관측되는 CartPole 환경"에서
# LSTM Actor-Critic 모델을 학습시키고,
# "좋은 성능이 나오는 학습 조건"을 찾기 위한 실험 코드다.
#
# 쉽게 말하면:
# 1) 기본 RL 모델(Actor-Critic + LSTM)을 만든다.
# 2) 학습 중 성능이 갑자기 떨어지면(collapse), 정책을 너무 흔들리지 않게 안정화한다.
# 3) 학습 중 성능이 갑자기 좋아지면(spike), 그 좋은 행동을 더 따라가게 만든다.
# 4) 이런 장치들의 강도(extra update, KL, SIL 등)를 바꿔가며
#    "어떤 설정이 제일 잘 학습되는지" 비교한다.
#
# ------------------------------------------------------------
# [좋은 모델을 어떻게 튜닝하는 코드인가?]
#
# 이 코드는 단순히 한 번 학습만 하는 게 아니라,
# "학습을 더 잘 되게 만드는 하이퍼파라미터 / 전략"을 비교하는 코드다.
#
# 튜닝 아이디어는 크게 3가지:
#
# (1) extra_updates_on_event
#     - 성능에 의미 있는 변화(collapse/spike)가 생겼을 때
#       actor를 몇 번 더 업데이트할지 정함
#     - 0,1,2,4,6처럼 바꿔가며 어떤 강도가 좋은지 실험
#
# (2) Phase Schedule (탐색 -> 수렴)
#     - 초반에는 많이 탐색하도록 entropy를 크게 유지
#     - 후반에는 탐색을 줄이고 안정적으로 수렴하도록 조정
#     - SIL은 후반에 줄이고, KL anchor는 후반에 강화
#
# (3) Ablation
#     - KL anchor 제거, SIL 제거, event update 제거 등
#       각 구성요소를 하나씩 빼 보면서
#       "어떤 장치가 실제로 성능에 기여하는지" 확인
#
# 즉, 이 코드는
# "좋은 RL 모델을 만드는 코드"이면서 동시에
# "좋은 학습 전략을 고르는 튜닝 실험 코드"다.
#
# ------------------------------------------------------------
# [서버에서 오래 돌릴 때 중요한 점]
#
# 이 코드에 체크포인트(resume)가 들어가 있어도,
# "컴퓨터를 꺼도 서버에서 계속 도는 것"은 실행 방식이 따로 중요하다.
#
# 예:
#   nohup python ablate_runner.py --device cuda:0 --resume > train.log 2>&1 &
#
# 또는 tmux / screen 사용.
#
# 정리:
# - 서버에서 계속 돌아가게 하는 것  -> nohup / tmux / screen
# - 중간에 끊겨도 이어서 하는 것   -> 이 코드의 checkpoint + --resume
#
# ------------------------------------------------------------
# [수정 사항]
#
# - notebook/ipykernel의 -f 인자 무시: parse_known_args()
# - AMP(mixed precision) + GradScaler (CUDA에서만)
# - torch.compile 옵션 반영
# - pin_memory 관련 numpy -> torch 변환 수정
# - full checkpoint 저장/로드 추가
# - --resume 옵션 추가
# - compiled model도 안전하게 state_dict 저장/로드 가능하도록 보완
#
# ============================================================

import os
import time
import json
import random
import argparse
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Bernoulli

# Excel
try:
    from openpyxl import Workbook
    from openpyxl.utils import get_column_letter
    OPENPYXL_OK = True
except Exception:
    OPENPYXL_OK = False

# TensorBoard
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_OK = True
except Exception:
    TENSORBOARD_OK = False


# =========================
# Base hyperparameters
# =========================
GAMMA = 0.99

ACTOR_HIDDEN = 256
CRITIC_HIDDEN = 256
LSTM_LAYERS = 1
LSTM_DROPOUT = 0.1

LR_ACTOR = 2e-4
LR_CRITIC = 2e-4
CLIP_NORM = 0.5

USE_HUBER_VALUE_LOSS = True

# Catch & Climb
BASELINE_BETA = 0.90
EPS = 1e-6

# Aux weights (base)
AUX_KL_COEF = 0.50
AUX_KL_MAX = 5.0

SIL_COEF = 0.50
SIL_MAX = 5.0

# Extra updates sweep candidates (FULL에서 사용)
EXTRA_UPDATES_SWEEP = [0, 1, 2, 4, 6]
EXTRA_UPDATES_CAP = 6

# Event deadzone: 너무 자잘한 변동은 이벤트로 치지 않음
EVENT_MIN_INT = 0.04

# Anchor EMA update
ANCHOR_EMA_BETA = 0.995

# Phase schedule (탐색 -> 수렴)
SCHEDULE_ON = True
SCHEDULE_START_FRAC = 0.60
ENT_COEF_START = 0.010
ENT_COEF_END = 0.003
SIL_SCALE_END = 0.50
KL_SCALE_END = 1.50

# Env / solve criterion
ENV_ID = "CartPole-v1"
MAX_STEPS_PER_EP = 500
SOLVE_AVG100 = 475.0


# =========================
# Repro / device
# =========================
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# =========================
# Tensor helpers
# =========================
def numpy_to_torch(
    arr: np.ndarray,
    dtype: torch.dtype,
    device: torch.device,
    pin_memory: bool = False,
    unsqueeze0: bool = False,
    view_shape: Optional[Tuple[int, ...]] = None,
) -> torch.Tensor:
    """
    numpy -> torch 변환 헬퍼

    주의:
    - torch.tensor(np_arr, pin_memory=True)는 에러 날 수 있음
    - 안전하게 torch.from_numpy(...).pin_memory() 사용
    """
    if not isinstance(arr, np.ndarray):
        arr = np.asarray(arr)

    if not arr.flags["C_CONTIGUOUS"]:
        arr = np.ascontiguousarray(arr)

    t = torch.from_numpy(arr)

    if t.dtype != dtype:
        t = t.to(dtype=dtype)

    if pin_memory and device.type == "cuda":
        t = t.pin_memory()

    if unsqueeze0:
        t = t.unsqueeze(0)

    if view_shape is not None:
        t = t.view(*view_shape)

    return t.to(device, non_blocking=(pin_memory and device.type == "cuda"))


# =========================
# Networks
# =========================
class PolicyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 64)
        self.lstm = nn.LSTM(
            64,
            ACTOR_HIDDEN,
            num_layers=LSTM_LAYERS,
            dropout=(LSTM_DROPOUT if LSTM_LAYERS >= 2 else 0.0),
            batch_first=True,
        )
        self.fc2 = nn.Linear(ACTOR_HIDDEN, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, hidden):
        x = self.relu(self.fc1(x))
        x, hidden = self.lstm(x, hidden)
        x = self.relu(x)
        x = self.sigmoid(self.fc2(x))  # (B,T,1)
        return x, hidden

    @torch.no_grad()
    def select_action(self, state, hidden):
        prob, hidden = self.forward(state, hidden)
        b = Bernoulli(prob)
        action = b.sample()
        return int(action.item()), hidden


class ValueNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 64)
        self.lstm = nn.LSTM(
            64,
            CRITIC_HIDDEN,
            num_layers=LSTM_LAYERS,
            dropout=(LSTM_DROPOUT if LSTM_LAYERS >= 2 else 0.0),
            batch_first=True,
        )
        self.fc2 = nn.Linear(CRITIC_HIDDEN, 1)
        self.relu = nn.ReLU()

    def forward(self, x, hidden):
        x = self.relu(self.fc1(x))
        x, hidden = self.lstm(x, hidden)
        x = self.relu(x)
        x = self.fc2(x)
        return x, hidden


# =========================
# Helpers
# =========================
def unwrap_model(model: nn.Module) -> nn.Module:
    """
    torch.compile(model)된 경우 state_dict key가 꼬이지 않게
    원본 모듈(_orig_mod)이 있으면 그걸 사용.
    """
    return model._orig_mod if hasattr(model, "_orig_mod") else model


def model_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return unwrap_model(model).state_dict()


def load_model_state_dict(model: nn.Module, state_dict: Dict[str, torch.Tensor], strict: bool = True):
    unwrap_model(model).load_state_dict(state_dict, strict=strict)


def obs_to_partial(obs):
    return np.array([obs[0], obs[2]], dtype=np.float32)


def bernoulli_kl(p, q, eps=1e-6):
    """
    KL(Bern(p) || Bern(q))
    """
    p = torch.clamp(p, eps, 1.0 - eps)
    q = torch.clamp(q, eps, 1.0 - eps)
    return p * torch.log(p / q) + (1.0 - p) * torch.log((1.0 - p) / (1.0 - q))


@torch.no_grad()
def forward_policy_probs(policy_net, states_tensor, device):
    a_hx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
    a_cx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
    prob, _ = policy_net(states_tensor, (a_hx, a_cx))
    return prob.squeeze(0)  # (T,1)


def compute_climb_signals(ep_reward, baseline_prev):
    denom = abs(baseline_prev) + 1.0
    collapse_raw = max(0.0, baseline_prev - ep_reward)
    spike_raw = max(0.0, ep_reward - baseline_prev)

    collapse_int = collapse_raw / (denom + EPS)
    spike_int = spike_raw / (denom + EPS)

    is_collapse = 1.0 if (ep_reward < baseline_prev and collapse_int >= EVENT_MIN_INT) else 0.0
    is_spike = 1.0 if (ep_reward > baseline_prev and spike_int >= EVENT_MIN_INT) else 0.0
    return is_collapse, is_spike, collapse_int, spike_int


def discounted_returns(rewards: List[float], gamma: float) -> np.ndarray:
    out = np.zeros(len(rewards), dtype=np.float32)
    R = 0.0
    for i in reversed(range(len(rewards))):
        R = gamma * R + rewards[i]
        out[i] = R
    return out


def compute_metrics(reward_list: List[float], solve_thr_avg100: float = SOLVE_AVG100) -> Dict[str, float]:
    r = np.asarray(reward_list, dtype=np.float32)
    auc_mean = float(r.mean())

    tail = r[-100:] if len(r) >= 100 else r
    last100_mean = float(tail.mean())
    last100_std = float(tail.std(ddof=0))

    first_solve = -1
    if len(r) >= 100:
        win = np.convolve(r, np.ones(100, dtype=np.float32) / 100.0, mode="valid")
        idx = np.where(win >= solve_thr_avg100)[0]
        if len(idx) > 0:
            first_solve = int(idx[0] + 100)

    eps = 1e-6
    stability = 1.0 - (last100_std / (abs(last100_mean) + eps))
    stability = float(max(-5.0, min(1.0, stability)))

    return {
        "auc_mean": auc_mean,
        "last100_mean": last100_mean,
        "last100_std": last100_std,
        "first_solve_ep(avg100>=thr)": float(first_solve),
        "stability_last100": stability,
    }


def linear_interp(a: float, b: float, t01: float) -> float:
    t01 = float(max(0.0, min(1.0, t01)))
    return a + (b - a) * t01


def phase_schedule(epoch: int, episodes: int) -> Dict[str, float]:
    """
    탐색 -> 수렴 스케줄
    """
    if (not SCHEDULE_ON) or episodes <= 1:
        return {"ent_coef": ENT_COEF_START, "sil_scale": 1.0, "kl_scale": 1.0}

    frac = epoch / float(episodes - 1)
    if frac <= SCHEDULE_START_FRAC:
        return {"ent_coef": ENT_COEF_START, "sil_scale": 1.0, "kl_scale": 1.0}

    t = (frac - SCHEDULE_START_FRAC) / (1.0 - SCHEDULE_START_FRAC)
    ent = linear_interp(ENT_COEF_START, ENT_COEF_END, t)
    sil_scale = linear_interp(1.0, SIL_SCALE_END, t)
    kl_scale = linear_interp(1.0, KL_SCALE_END, t)
    return {"ent_coef": ent, "sil_scale": sil_scale, "kl_scale": kl_scale}


# =========================
# Ablation config
# =========================
@dataclass
class Variant:
    name: str
    use_kl_anchor: bool = True
    use_sil: bool = True
    extra_on_collapse: bool = True
    extra_on_spike: bool = True
    use_entropy: bool = True
    use_adv_norm: bool = True


def get_variants() -> List[Variant]:
    return [
        Variant("FULL",       use_kl_anchor=True,  use_sil=True,  extra_on_collapse=True,  extra_on_spike=True,  use_entropy=True, use_adv_norm=True),
        Variant("NoSpike",    use_kl_anchor=True,  use_sil=False, extra_on_collapse=True,  extra_on_spike=False, use_entropy=True, use_adv_norm=True),
        Variant("NoEventUpd", use_kl_anchor=True,  use_sil=True,  extra_on_collapse=False, extra_on_spike=False, use_entropy=True, use_adv_norm=True),
        Variant("NoCollapse", use_kl_anchor=False, use_sil=True,  extra_on_collapse=False, extra_on_spike=True,  use_entropy=True, use_adv_norm=True),
        Variant("Vanilla",    use_kl_anchor=False, use_sil=False, extra_on_collapse=False, extra_on_spike=False, use_entropy=True, use_adv_norm=True),
    ]


# =========================
# Checkpoint utils
# =========================
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def get_ckpt_path(ckpt_dir: str, variant_name: str, eu: int, seed: int) -> str:
    ensure_dir(ckpt_dir)
    return os.path.join(ckpt_dir, f"{variant_name}_EU{eu}_seed{seed}_latest.pt")


def get_meta_path(ckpt_dir: str, variant_name: str, eu: int, seed: int) -> str:
    ensure_dir(ckpt_dir)
    return os.path.join(ckpt_dir, f"{variant_name}_EU{eu}_seed{seed}_meta.json")


def save_json(path: str, data: Dict[str, object]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path: str) -> Dict[str, object]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_checkpoint(
    path: str,
    epoch: int,
    policy: nn.Module,
    value: nn.Module,
    optim: torch.optim.Optimizer,
    value_optim: torch.optim.Optimizer,
    anchor_policy: nn.Module,
    baseline: float,
    ep_rewards: List[float],
    collapse_count: int,
    spike_count: int,
):
    ckpt = {
        "epoch": epoch,
        "policy": model_state_dict(policy),
        "value": model_state_dict(value),
        "optim": optim.state_dict(),
        "value_optim": value_optim.state_dict(),
        "anchor_policy": model_state_dict(anchor_policy),
        "baseline": baseline,
        "ep_rewards": ep_rewards,
        "collapse_count": collapse_count,
        "spike_count": spike_count,
    }
    torch.save(ckpt, path)


def load_checkpoint(
    path: str,
    policy: nn.Module,
    value: nn.Module,
    optim: torch.optim.Optimizer,
    value_optim: torch.optim.Optimizer,
    anchor_policy: nn.Module,
    device: torch.device,
):
    ckpt = torch.load(path, map_location=device)

    load_model_state_dict(policy, ckpt["policy"], strict=True)
    load_model_state_dict(value, ckpt["value"], strict=True)
    optim.load_state_dict(ckpt["optim"])
    value_optim.load_state_dict(ckpt["value_optim"])
    load_model_state_dict(anchor_policy, ckpt["anchor_policy"], strict=True)

    start_epoch = int(ckpt["epoch"]) + 1
    baseline = float(ckpt.get("baseline", 0.0))
    ep_rewards = list(ckpt.get("ep_rewards", []))
    collapse_count = int(ckpt.get("collapse_count", 0))
    spike_count = int(ckpt.get("spike_count", 0))

    return start_epoch, baseline, ep_rewards, collapse_count, spike_count


# =========================
# One run (one seed, one variant)
# =========================
def run_variant(
    variant: Variant,
    episodes: int,
    seed: int,
    extra_updates_on_event: int,
    log_dir_root: Optional[str],
    device: torch.device,
    amp: bool = True,
    save_policy_every: int = 0,
    verbose_every: int = 200,
    compile_models: bool = False,
    ckpt_dir: str = "./checkpoints",
    save_every: int = 200,
    resume: bool = False,
) -> Dict[str, object]:
    set_global_seed(seed)
    env = gym.make(ENV_ID)

    policy = PolicyNetwork().to(device)
    value = ValueNetwork().to(device)

    if compile_models:
        policy = maybe_compile(policy, True)
        value = maybe_compile(value, True)

    optim = torch.optim.Adam(unwrap_model(policy).parameters(), lr=LR_ACTOR)
    value_optim = torch.optim.Adam(unwrap_model(value).parameters(), lr=LR_CRITIC)

    anchor_policy = PolicyNetwork().to(device)
    load_model_state_dict(anchor_policy, model_state_dict(policy), strict=True)
    anchor_policy.eval()

    if compile_models:
        anchor_policy = maybe_compile(anchor_policy, True)

    use_amp = bool(amp) and (device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    baseline = 0.0
    start_epoch = 0
    ep_rewards: List[float] = []
    collapse_count = 0
    spike_count = 0

    ckpt_path = get_ckpt_path(ckpt_dir, variant.name, extra_updates_on_event, seed)
    meta_path = get_meta_path(ckpt_dir, variant.name, extra_updates_on_event, seed)

    if resume and os.path.exists(ckpt_path):
        print(f"[resume] loading checkpoint: {ckpt_path}")
        start_epoch, baseline, ep_rewards, collapse_count, spike_count = load_checkpoint(
            ckpt_path, policy, value, optim, value_optim, anchor_policy, device
        )
        print(f"[resume] resumed from epoch={start_epoch} / total={episodes}")

    # TensorBoard
    writer = None
    run_tag = f"{variant.name}_EU{extra_updates_on_event}_seed{seed}"
    if log_dir_root is not None and TENSORBOARD_OK:
        ensure_dir(log_dir_root)
        writer = SummaryWriter(os.path.join(log_dir_root, run_tag))

    pin_mem = (device.type == "cuda")

    if start_epoch >= episodes:
        print(f"[skip] already finished: {variant.name} EU={extra_updates_on_event} seed={seed}")
        metrics = compute_metrics(ep_rewards, SOLVE_AVG100)
        out = {
            "variant": variant.name,
            "seed": seed,
            "episodes": episodes,
            "extra_updates_on_event": int(extra_updates_on_event),
            "collapse_count": float(collapse_count),
            "spike_count": float(spike_count),
            **metrics,
        }
        if writer is not None:
            writer.close()
        env.close()
        return out

    for epoch in range(start_epoch, episodes):
        sched = phase_schedule(epoch, episodes)
        ent_coef = sched["ent_coef"]
        sil_scale = sched["sil_scale"]
        kl_scale = sched["kl_scale"]

        obs, _info = env.reset(seed=seed * 100000 + epoch)
        state = obs_to_partial(obs)
        episode_reward = 0.0

        a_hx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
        a_cx = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)

        rewards, actions, states = [], [], []

        for _t in range(MAX_STEPS_PER_EP):
            states.append(state.copy())

            state_t = torch.as_tensor(state, dtype=torch.float32, device=device).view(1, 1, 2)
            action, (a_hx, a_cx) = policy.select_action(state_t, (a_hx, a_cx))
            actions.append(action)

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            state = obs_to_partial(next_obs)
            episode_reward += float(reward)
            rewards.append(float(reward))

            if done:
                break

        ep_rewards.append(float(episode_reward))

        returns = discounted_returns(rewards, GAMMA)
        mean, std = returns.mean(), returns.std()
        std = std if std > 1e-8 else 1.0
        returns_norm = (returns - mean) / std

        states_np = np.asarray(states, dtype=np.float32)
        actions_np = np.asarray(actions, dtype=np.float32)
        returns_np = np.asarray(returns_norm, dtype=np.float32)

        states_tensor = numpy_to_torch(
            states_np, dtype=torch.float32, device=device, pin_memory=pin_mem, unsqueeze0=True
        )
        actions_tensor = numpy_to_torch(
            actions_np, dtype=torch.float32, device=device, pin_memory=pin_mem, view_shape=(-1, 1)
        )
        returns_tensor = numpy_to_torch(
            returns_np, dtype=torch.float32, device=device, pin_memory=pin_mem, view_shape=(-1, 1)
        )

        baseline_prev = float(baseline)
        is_collapse, is_spike, collapse_int, spike_int = compute_climb_signals(episode_reward, baseline_prev)
        baseline = BASELINE_BETA * baseline + (1.0 - BASELINE_BETA) * float(episode_reward)

        if is_collapse:
            collapse_count += 1
        if is_spike:
            spike_count += 1

        kl_w = 0.0
        sil_w = 0.0

        if variant.use_kl_anchor and is_collapse:
            kl_w = min(AUX_KL_MAX, AUX_KL_COEF * (1.0 + 5.0 * collapse_int))
            kl_w *= kl_scale

        if variant.use_sil and is_spike:
            sil_w = min(SIL_MAX, SIL_COEF * (1.0 + 5.0 * spike_int))
            sil_w *= sil_scale

        extra_updates = 0
        if is_collapse and variant.extra_on_collapse:
            extra_updates = max(extra_updates, extra_updates_on_event)
        if is_spike and variant.extra_on_spike:
            extra_updates = max(extra_updates, extra_updates_on_event)
        extra_updates = int(min(EXTRA_UPDATES_CAP, extra_updates))

        with torch.no_grad():
            c_hx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
            c_cx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
            v, _ = value(states_tensor, (c_hx, c_cx))
            v = v.squeeze(0)
            advantage = returns_tensor - v
            if variant.use_adv_norm:
                advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)

        anchor_prob = None
        if variant.use_kl_anchor:
            with torch.no_grad():
                anchor_prob = forward_policy_probs(anchor_policy, states_tensor, device)

        def actor_step() -> Tuple[float, float, float, float, float]:
            a_hx0 = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)
            a_cx0 = torch.zeros((LSTM_LAYERS, 1, ACTOR_HIDDEN), device=device)

            with torch.amp.autocast("cuda", enabled=use_amp):
                prob, _ = policy(states_tensor, (a_hx0, a_cx0))
                prob = prob.squeeze(0)

                dist = Bernoulli(prob)
                log_prob = dist.log_prob(actions_tensor)
                entropy = dist.entropy().mean()

                ent_term = (ent_coef * entropy) if variant.use_entropy else 0.0
                base_loss = -(log_prob * advantage.detach()).mean() - ent_term

                kl_loss = 0.0
                if kl_w > 0.0 and anchor_prob is not None:
                    kl = bernoulli_kl(prob, anchor_prob).mean()
                    kl_loss = float(kl_w) * kl

                sil_loss = 0.0
                if sil_w > 0.0:
                    pos_adv = torch.clamp(advantage.detach(), min=0.0)
                    sil_loss = float(sil_w) * (-(log_prob * pos_adv).mean())

                loss = base_loss + kl_loss + sil_loss

            optim.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(unwrap_model(policy).parameters(), CLIP_NORM)
            scaler.step(optim)
            scaler.update()

            def f(x):
                return float(x.item()) if hasattr(x, "item") else float(x)

            return f(loss), f(base_loss), f(kl_loss), f(sil_loss), f(entropy)

        actor_total, actor_base, actor_kl, actor_sil, entropy_v = actor_step()
        for _ in range(extra_updates):
            actor_total, actor_base, actor_kl, actor_sil, entropy_v = actor_step()

        c_hx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)
        c_cx = torch.zeros((LSTM_LAYERS, 1, CRITIC_HIDDEN), device=device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            v_pred, _ = value(states_tensor, (c_hx, c_cx))
            v_pred = v_pred.squeeze(0)
            if USE_HUBER_VALUE_LOSS:
                value_loss = F.smooth_l1_loss(v_pred, returns_tensor)
            else:
                value_loss = F.mse_loss(v_pred, returns_tensor)

        value_optim.zero_grad(set_to_none=True)
        scaler.scale(value_loss).backward()
        scaler.unscale_(value_optim)
        torch.nn.utils.clip_grad_norm_(unwrap_model(value).parameters(), CLIP_NORM)
        scaler.step(value_optim)
        scaler.update()

        if variant.use_kl_anchor:
            with torch.no_grad():
                anchor_sd = model_state_dict(anchor_policy)
                policy_sd = model_state_dict(policy)
                for k in anchor_sd.keys():
                    anchor_sd[k].mul_(ANCHOR_EMA_BETA).add_(policy_sd[k], alpha=(1.0 - ANCHOR_EMA_BETA))
                load_model_state_dict(anchor_policy, anchor_sd, strict=True)
            anchor_policy.eval()

        if writer is not None:
            writer.add_scalar("episode_reward", episode_reward, epoch)
            writer.add_scalar("baseline/ema", baseline, epoch)
            writer.add_scalar("catch/is_collapse", is_collapse, epoch)
            writer.add_scalar("catch/is_spike", is_spike, epoch)
            writer.add_scalar("catch/collapse_int", collapse_int, epoch)
            writer.add_scalar("catch/spike_int", spike_int, epoch)
            writer.add_scalar("sched/ent_coef", ent_coef, epoch)
            writer.add_scalar("sched/sil_scale", sil_scale, epoch)
            writer.add_scalar("sched/kl_scale", kl_scale, epoch)
            writer.add_scalar("catch/kl_weight", kl_w, epoch)
            writer.add_scalar("catch/sil_weight", sil_w, epoch)
            writer.add_scalar("catch/extra_updates", extra_updates, epoch)
            writer.add_scalar("loss/actor_total", actor_total, epoch)
            writer.add_scalar("loss/actor_base", actor_base, epoch)
            writer.add_scalar("loss/actor_kl", actor_kl, epoch)
            writer.add_scalar("loss/actor_sil", actor_sil, epoch)
            writer.add_scalar("loss/value", float(value_loss.item()), epoch)
            writer.add_scalar("stats/entropy", entropy_v, epoch)

        if (epoch % verbose_every) == 0:
            tag = "COLLAPSE" if is_collapse else ("SPIKE" if is_spike else "normal")
            print(
                f"[{variant.name} EU={extra_updates_on_event} seed={seed}] "
                f"ep {epoch:05d} | R {episode_reward:6.1f} | ema {baseline:7.2f} | "
                f"{tag} | extra={extra_updates}"
            )

        if save_policy_every and (epoch + 1) % save_policy_every == 0:
            torch.save(
                model_state_dict(policy),
                os.path.join(
                    ckpt_dir,
                    f"policy_{variant.name}_EU{extra_updates_on_event}_seed{seed}_ep{epoch+1}.pt",
                ),
            )

        if save_every and (epoch + 1) % save_every == 0:
            save_checkpoint(
                ckpt_path,
                epoch,
                policy,
                value,
                optim,
                value_optim,
                anchor_policy,
                baseline,
                ep_rewards,
                collapse_count,
                spike_count,
            )
            save_json(meta_path, {
                "variant": variant.name,
                "seed": seed,
                "episodes_total": episodes,
                "last_saved_epoch": epoch,
                "extra_updates_on_event": extra_updates_on_event,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            })
            print(f"[ckpt] saved: {ckpt_path} @ ep {epoch+1}")

    save_checkpoint(
        ckpt_path,
        episodes - 1,
        policy,
        value,
        optim,
        value_optim,
        anchor_policy,
        baseline,
        ep_rewards,
        collapse_count,
        spike_count,
    )
    save_json(meta_path, {
        "variant": variant.name,
        "seed": seed,
        "episodes_total": episodes,
        "last_saved_epoch": episodes - 1,
        "extra_updates_on_event": extra_updates_on_event,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": "finished",
    })
    print(f"[ckpt] final saved: {ckpt_path}")

    if writer is not None:
        writer.close()
    env.close()

    metrics = compute_metrics(ep_rewards, SOLVE_AVG100)
    out = {
        "variant": variant.name,
        "seed": seed,
        "episodes": episodes,
        "extra_updates_on_event": int(extra_updates_on_event),
        "collapse_count": float(collapse_count),
        "spike_count": float(spike_count),
        **metrics,
    }
    return out


# =========================
# Aggregation
# =========================
def aggregate_across_seeds(results: List[Dict[str, object]]) -> List[Dict[str, object]]:
    by: Dict[str, List[Dict[str, object]]] = {}
    for r in results:
        key = f"{r['variant']}_EU{r.get('extra_updates_on_event', 0)}"
        by.setdefault(key, []).append(r)

    keys = [
        "auc_mean",
        "last100_mean",
        "last100_std",
        "first_solve_ep(avg100>=thr)",
        "stability_last100",
        "collapse_count",
        "spike_count",
    ]

    agg = []
    for kname, rs in by.items():
        out = {"group": kname, "n_seeds": len(rs)}
        out["variant"] = rs[0]["variant"]
        out["extra_updates_on_event"] = int(rs[0].get("extra_updates_on_event", 0))
        for k in keys:
            vals = [float(x[k]) for x in rs]
            out[k] = float(np.mean(vals))
        agg.append(out)

    agg.sort(key=lambda x: x["last100_mean"], reverse=True)
    return agg


def aggregate_by_variant_only(results: List[Dict[str, object]]) -> List[Dict[str, object]]:
    by: Dict[str, List[Dict[str, object]]] = {}
    for r in results:
        by.setdefault(str(r["variant"]), []).append(r)

    keys = [
        "auc_mean",
        "last100_mean",
        "last100_std",
        "first_solve_ep(avg100>=thr)",
        "stability_last100",
        "collapse_count",
        "spike_count",
    ]

    agg = []
    for v, rs in by.items():
        out = {"variant": v, "n_seeds": len(rs)}
        out["extra_updates_on_event"] = int(rs[0].get("extra_updates_on_event", 0))
        for k in keys:
            vals = [float(x[k]) for x in rs]
            out[k] = float(np.mean(vals))
        agg.append(out)

    agg.sort(key=lambda x: x["last100_mean"], reverse=True)
    return agg


# =========================
# Excel writer
# =========================
def autosize_worksheet(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                v = "" if cell.value is None else str(cell.value)
                max_len = max(max_len, len(v))
            except Exception:
                pass
        ws.column_dimensions[col_letter].width = min(60, max(10, max_len + 2))


def write_sheet(ws, rows: List[Dict[str, object]], title: str = ""):
    if title:
        ws.append([title])
        ws.append([])

    if not rows:
        ws.append(["(no rows)"])
        return

    cols = sorted({k for r in rows for k in r.keys()})
    ws.append(cols)
    for r in rows:
        ws.append([r.get(c, "") for c in cols])

    autosize_worksheet(ws)


def save_to_excel(
    path: str,
    sweep_raw: List[Dict[str, object]],
    sweep_summary: List[Dict[str, object]],
    ablation_raw: List[Dict[str, object]],
    ablation_summary: List[Dict[str, object]],
):
    if not OPENPYXL_OK:
        raise RuntimeError("openpyxl is not available. Please `pip install openpyxl`.")

    wb = Workbook()
    ws0 = wb.active
    ws0.title = "readme"
    ws0.append(["test_6 results workbook"])
    ws0.append(["Sheets: sweep_raw, sweep_summary, ablation_raw, ablation_summary"])
    ws0.append(["Saved at:", time.strftime("%Y-%m-%d %H:%M:%S")])

    ws = wb.create_sheet("sweep_raw")
    write_sheet(ws, sweep_raw, "FULL extra_updates sweep (raw per seed)")

    ws = wb.create_sheet("sweep_summary")
    write_sheet(ws, sweep_summary, "FULL extra_updates sweep (summary across seeds)")

    ws = wb.create_sheet("ablation_raw")
    write_sheet(ws, ablation_raw, "Ablation (raw per seed)")

    ws = wb.create_sheet("ablation_summary")
    write_sheet(ws, ablation_summary, "Ablation (summary across seeds)")

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    wb.save(path)


# =========================
# Selection logic for best EU
# =========================
def pick_best_extra_updates(sweep_summary: List[Dict[str, object]]) -> int:
    """
    우선순위:
    (1) solve 여부
    (2) AUC
    (3) last100_mean
    (4) stability
    """
    def key_fn(r):
        first = float(r.get("first_solve_ep(avg100>=thr)", -1))
        solved = 1 if first > 0 else 0
        return (
            solved,
            float(r.get("auc_mean", 0.0)),
            float(r.get("last100_mean", 0.0)),
            float(r.get("stability_last100", -999.0)),
            -first if solved else 0.0,
        )

    best = max(sweep_summary, key=key_fn)
    return int(best["extra_updates_on_event"])


# =========================
# Args / compile
# =========================
def parse_args():
    p = argparse.ArgumentParser(add_help=True)

    p.add_argument("--device", type=str, default=None, help="e.g. cuda:0 / cuda:1 / cpu. default: auto")
    p.add_argument("--amp", action="store_true", help="use mixed precision on cuda")
    p.add_argument("--no-amp", dest="amp", action="store_false")
    p.set_defaults(amp=True)

    p.add_argument("--compile", action="store_true", help="torch.compile models (PyTorch 2.x)")
    p.add_argument("--logdir", type=str, default="./runs_ablation", help="TensorBoard root (set '' to disable)")
    p.add_argument("--xlsx", type=str, default="./test_6.xlsx", help="output xlsx path")

    p.add_argument("--save-policy-every", type=int, default=0, help="save policy-only weights every N episodes")
    p.add_argument("--verbose-every", type=int, default=200)

    p.add_argument("--episodes-sweep", type=int, default=5000)
    p.add_argument("--episodes-ablation", type=int, default=5000)
    p.add_argument("--seeds", type=str, default="0,1,2,3,4", help="comma-separated seeds")

    # checkpoint / resume
    p.add_argument("--ckpt-dir", type=str, default="./checkpoints", help="checkpoint directory")
    p.add_argument("--save-every", type=int, default=200, help="save full checkpoint every N episodes")
    p.add_argument("--resume", action="store_true", help="resume from latest checkpoint if exists")

    # Jupyter(ipykernel)에서 자동으로 붙는 -f 같은 인자 무시
    args, _unknown = p.parse_known_args()
    return args


def maybe_compile(model: nn.Module, enabled: bool) -> nn.Module:
    if not enabled:
        return model
    try:
        return torch.compile(model)
    except Exception as e:
        print("[warn] torch.compile failed -> fallback. err:", repr(e))
        return model


# =========================
# Main
# =========================
def main():
    args = parse_args()

    if args.device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(args.device)

    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    print("device =", device, "| amp =", (args.amp and device.type == "cuda"), "| compile =", args.compile)

    SEEDS = [int(x.strip()) for x in args.seeds.split(",") if x.strip() != ""]
    EPISODES_SWEEP = int(args.episodes_sweep)
    EPISODES_ABLATION = int(args.episodes_ablation)

    LOG_ROOT = args.logdir if (args.logdir is not None and args.logdir.strip() != "") else None
    SAVE_POLICY_EVERY = int(args.save_policy_every)
    VERBOSE_EVERY = int(args.verbose_every)

    # ---------- (1) FULL: extra_updates sweep ----------
    sweep_raw: List[Dict[str, object]] = []
    full = Variant("FULL", use_kl_anchor=True, use_sil=True, extra_on_collapse=True, extra_on_spike=True)

    print("\n\n===== [1] FULL extra_updates sweep =====")
    for eu in EXTRA_UPDATES_SWEEP:
        for s in SEEDS:
            print(f"\n--- SWEEP RUN: FULL EU={eu} seed={s} episodes={EPISODES_SWEEP} ---")
            res = run_variant(
                variant=full,
                episodes=EPISODES_SWEEP,
                seed=s,
                extra_updates_on_event=eu,
                log_dir_root=LOG_ROOT,
                device=device,
                amp=args.amp,
                save_policy_every=SAVE_POLICY_EVERY,
                verbose_every=VERBOSE_EVERY,
                compile_models=args.compile,
                ckpt_dir=args.ckpt_dir,
                save_every=args.save_every,
                resume=args.resume,
            )
            sweep_raw.append(res)

    sweep_summary = aggregate_across_seeds(sweep_raw)

    best_eu = pick_best_extra_updates(sweep_summary)
    print("\n===== Sweep summary (top 5 by last100_mean) =====")
    for r in sweep_summary[:5]:
        print(r)
    print(f"\n>>> Picked best extra_updates_on_event for FINAL ABLATION: {best_eu}")

    # ---------- (2) Final Ablation with best EU ----------
    print("\n\n===== [2] FINAL ABLATION (best EU) =====")
    variants = get_variants()
    ablation_raw: List[Dict[str, object]] = []
    for v in variants:
        for s in SEEDS:
            print(f"\n=== ABLATION RUN: {v.name} EU={best_eu} seed={s} episodes={EPISODES_ABLATION} ===")
            res = run_variant(
                variant=v,
                episodes=EPISODES_ABLATION,
                seed=s,
                extra_updates_on_event=best_eu,
                log_dir_root=LOG_ROOT,
                device=device,
                amp=args.amp,
                save_policy_every=SAVE_POLICY_EVERY,
                verbose_every=VERBOSE_EVERY,
                compile_models=args.compile,
                ckpt_dir=args.ckpt_dir,
                save_every=args.save_every,
                resume=args.resume,
            )
            ablation_raw.append(res)

    ablation_summary = aggregate_by_variant_only(ablation_raw)

    # ---------- Save Excel ----------
    out_xlsx = args.xlsx
    save_to_excel(out_xlsx, sweep_raw, sweep_summary, ablation_raw, ablation_summary)
    print(f"\nSaved Excel: {out_xlsx}")

    # ---------- Print quick table ----------
    print("\n\n========== FINAL ABLATION SUMMARY (mean across seeds) ==========")
    cols = [
        "variant", "n_seeds",
        "auc_mean", "last100_mean", "last100_std",
        "first_solve_ep(avg100>=thr)",
        "stability_last100",
        "collapse_count", "spike_count",
        "extra_updates_on_event",
    ]

    def fmt_row(r):
        out = {}
        for c in cols:
            out[c] = r.get(c, "")
        for k in ["auc_mean", "last100_mean", "last100_std", "stability_last100"]:
            if k in out and out[k] != "":
                out[k] = f"{float(out[k]):.3f}"
        if "first_solve_ep(avg100>=thr)" in out and out["first_solve_ep(avg100>=thr)"] != "":
            out["first_solve_ep(avg100>=thr)"] = int(float(out["first_solve_ep(avg100>=thr)"]))
        for k in ["collapse_count", "spike_count"]:
            if k in out and out[k] != "":
                out[k] = f"{float(out[k]):.1f}"
        return out

    pretty = [fmt_row(r) for r in ablation_summary]
    widths = {c: max(len(c), max(len(str(pr.get(c, ""))) for pr in pretty)) for c in cols}
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for pr in pretty:
        print(" | ".join(str(pr.get(c, "")).ljust(widths[c]) for c in cols))

    if LOG_ROOT is not None and TENSORBOARD_OK:
        print(f"\nTensorBoard: tensorboard --logdir {LOG_ROOT}")

    print("\n[server run example]")
    print(
        "nohup python ablate_runner.py "
        "--device cuda:0 --resume "
        "--save-every 200 --save-policy-every 200 "
        "--ckpt-dir ./checkpoints "
        "--logdir ./runs_ablation "
        "--xlsx ./test_6.xlsx "
        "> train.log 2>&1 &"
    )


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# robust_acrobot_mountaincar_ddqn.py
# ------------------------------------------------------------
# 목표
# - Acrobot-v1 / MountainCar-v0 를 안정적으로 학습
# - Dueling Double DQN + Prioritized Replay + n-step
# - sparse reward 환경 대응
# - train reward shaping / eval raw reward 분리
# - GPU 우선 사용
# - checkpoint / resume / csv / json 저장
# ============================================================

import os
import gc
import csv
import json
import math
import time
import random
import signal
import argparse
from dataclasses import dataclass, asdict
from collections import deque
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Global
# ============================================================
STOP_REQUESTED = False


def _signal_handler(signum, frame):
    global STOP_REQUESTED
    STOP_REQUESTED = True
    print(f"\n[signal] received signal={signum}. stopping at next safe point...")


signal.signal(signal.SIGINT, _signal_handler)
signal.signal(signal.SIGTERM, _signal_handler)


# ============================================================
# Utils
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def safe_mean(xs, default=0.0):
    return float(np.mean(xs)) if len(xs) > 0 else float(default)


def linear_schedule(start: float, end: float, progress: float) -> float:
    progress = float(np.clip(progress, 0.0, 1.0))
    return float(start + (end - start) * progress)


# ============================================================
# Running Mean Std
# ============================================================
class RunningMeanStd:
    def __init__(self, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = 1e-4

    def update(self, x: np.ndarray):
        x = np.asarray(x, dtype=np.float64)
        if x.ndim == 1:
            batch_mean = x
            batch_var = np.zeros_like(x)
            batch_count = 1
        else:
            batch_mean = x.mean(axis=0)
            batch_var = x.var(axis=0)
            batch_count = x.shape[0]
        self._update_from_moments(batch_mean, batch_var, batch_count)

    def _update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        total_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / total_count

        m_a = self.var * self.count
        m_b = batch_var * batch_count
        m2 = m_a + m_b + (delta ** 2) * self.count * batch_count / total_count
        new_var = m2 / total_count

        self.mean = new_mean
        self.var = np.maximum(new_var, 1e-8)
        self.count = total_count

    def normalize(self, x: np.ndarray, clip: float = 10.0) -> np.ndarray:
        x = (x - self.mean) / np.sqrt(self.var + 1e-8)
        return np.clip(x, -clip, clip)

    def state_dict(self):
        return {"mean": self.mean, "var": self.var, "count": self.count}

    def load_state_dict(self, d):
        self.mean = d["mean"]
        self.var = d["var"]
        self.count = d["count"]


# ============================================================
# Presets
# ============================================================
@dataclass
class Preset:
    name: str
    env_id: str
    total_steps: int
    warmup_steps: int
    buffer_size: int
    batch_size: int
    hidden_dim: int
    lr: float
    min_lr: float
    gamma: float
    n_step: int
    target_update_interval: int
    tau: float
    eps_start: float
    eps_end: float
    eps_decay_steps: int
    per_alpha: float
    per_beta_start: float
    per_beta_end: float
    train_freq: int
    gradient_steps: int
    max_grad_norm: float
    eval_interval_steps: int
    eval_episodes: int
    solve_threshold: float
    solve_window: int
    use_obs_norm: bool


PRESETS = {
    "acrobot_best": Preset(
        name="acrobot_best",
        env_id="Acrobot-v1",
        total_steps=450_000,
        warmup_steps=10_000,
        buffer_size=250_000,
        batch_size=256,
        hidden_dim=256,
        lr=2.5e-4,
        min_lr=8.0e-5,
        gamma=0.99,
        n_step=5,
        target_update_interval=1,
        tau=0.01,
        eps_start=1.0,
        eps_end=0.03,
        eps_decay_steps=180_000,
        per_alpha=0.6,
        per_beta_start=0.4,
        per_beta_end=1.0,
        train_freq=1,
        gradient_steps=1,
        max_grad_norm=10.0,
        eval_interval_steps=10_000,
        eval_episodes=10,
        solve_threshold=-90.0,
        solve_window=100,
        use_obs_norm=True,
    ),
    "mountaincar_best": Preset(
        name="mountaincar_best",
        env_id="MountainCar-v0",
        total_steps=350_000,
        warmup_steps=8_000,
        buffer_size=200_000,
        batch_size=256,
        hidden_dim=256,
        lr=2.0e-4,
        min_lr=6.0e-5,
        gamma=0.997,
        n_step=3,
        target_update_interval=1,
        tau=0.01,
        eps_start=1.0,
        eps_end=0.02,
        eps_decay_steps=140_000,
        per_alpha=0.6,
        per_beta_start=0.4,
        per_beta_end=1.0,
        train_freq=1,
        gradient_steps=1,
        max_grad_norm=10.0,
        eval_interval_steps=8_000,
        eval_episodes=10,
        solve_threshold=-110.0,
        solve_window=100,
        use_obs_norm=True,
    ),
}


# ============================================================
# Reward shaping (train only)
# ============================================================
def train_reward_shaping(env_id: str, obs: np.ndarray, next_obs: np.ndarray, raw_reward: float, done: bool) -> float:
    shaped = float(raw_reward)

    if env_id == "MountainCar-v0":
        pos, vel = next_obs
        prev_pos, prev_vel = obs

        progress = 6.0 * (pos - prev_pos)
        speed_bonus = 1.5 * abs(vel)
        uphill_bonus = 0.5 * max(0.0, pos + 0.5)

        terminal_bonus = 0.0
        if done and pos >= 0.5:
            terminal_bonus = 120.0

        shaped = raw_reward + progress + speed_bonus + uphill_bonus + terminal_bonus

    elif env_id == "Acrobot-v1":
        c1, s1, c2, s2, d1, d2 = next_obs
        pc1, ps1, pc2, ps2, pd1, pd2 = obs

        # tip height surrogate
        cur_height_like = -(c1 + c1 * c2 - s1 * s2)
        prev_height_like = -(pc1 + pc1 * pc2 - ps1 * ps2)

        height_progress = 3.5 * (cur_height_like - prev_height_like)
        height_bonus = 0.5 * cur_height_like
        vel_penalty = -0.002 * (abs(d1) + abs(d2))

        terminal_bonus = 0.0
        if done:
            terminal_bonus = 25.0

        shaped = raw_reward + height_progress + height_bonus + vel_penalty + terminal_bonus

    return float(shaped)


# ============================================================
# Dueling Q Network
# ============================================================
class DuelingQNetwork(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.adv_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=math.sqrt(2))
                nn.init.constant_(m.bias, 0.0)
        nn.init.orthogonal_(self.value_stream[-1].weight, gain=1.0)
        nn.init.orthogonal_(self.adv_stream[-1].weight, gain=0.01)

    def forward(self, x):
        z = self.feature(x)
        v = self.value_stream(z)
        a = self.adv_stream(z)
        q = v + (a - a.mean(dim=-1, keepdim=True))
        return q


# ============================================================
# Prioritized Replay with n-step
# ============================================================
class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, obs_dim: int, alpha: float = 0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.pos = 0
        self.size = 0

        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros((capacity,), dtype=np.int64)
        self.rewards = np.zeros((capacity,), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.dones = np.zeros((capacity,), dtype=np.float32)
        self.priorities = np.zeros((capacity,), dtype=np.float32)

    def add(self, obs, action, reward, next_obs, done):
        self.obs[self.pos] = obs
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.next_obs[self.pos] = next_obs
        self.dones[self.pos] = float(done)

        max_prio = self.priorities.max() if self.size > 0 else 1.0
        self.priorities[self.pos] = max_prio

        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int, beta: float):
        if self.size == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.size]

        probs = prios ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(self.size, batch_size, p=probs)
        weights = (self.size * probs[indices]) ** (-beta)
        weights /= weights.max()

        batch = dict(
            obs=self.obs[indices],
            actions=self.actions[indices],
            rewards=self.rewards[indices],
            next_obs=self.next_obs[indices],
            dones=self.dones[indices],
            indices=indices,
            weights=weights.astype(np.float32),
        )
        return batch

    def update_priorities(self, indices, priorities):
        priorities = np.asarray(priorities, dtype=np.float32)
        self.priorities[indices] = np.maximum(priorities, 1e-6)

    def __len__(self):
        return self.size


class NStepTransitionAccumulator:
    def __init__(self, n_step: int, gamma: float):
        self.n_step = n_step
        self.gamma = gamma
        self.buffer = deque(maxlen=n_step)

    def reset(self):
        self.buffer.clear()

    def append(self, transition):
        self.buffer.append(transition)

    def is_ready(self):
        return len(self.buffer) >= self.n_step

    def pop_n_step(self):
        reward = 0.0
        next_obs = None
        done = False

        obs, action, _, _, _ = self.buffer[0]

        for i, (_, _, r, nobs, d) in enumerate(self.buffer):
            reward += (self.gamma ** i) * r
            next_obs = nobs
            done = d
            if d:
                break

        self.buffer.popleft()
        return obs, action, reward, next_obs, done

    def flush_remaining(self):
        items = []
        while len(self.buffer) > 0:
            reward = 0.0
            next_obs = None
            done = False
            obs, action, _, _, _ = self.buffer[0]

            for i, (_, _, r, nobs, d) in enumerate(self.buffer):
                reward += (self.gamma ** i) * r
                next_obs = nobs
                done = d
                if d:
                    break

            self.buffer.popleft()
            items.append((obs, action, reward, next_obs, done))
        return items


# ============================================================
# Checkpoint
# ============================================================
def save_checkpoint(path, q_net, target_net, optimizer, obs_rms, state):
    obj = {
        "q_net": q_net.state_dict(),
        "target_net": target_net.state_dict(),
        "optimizer": optimizer.state_dict(),
        "obs_rms": obs_rms.state_dict() if obs_rms is not None else None,
        "state": state,
    }
    torch.save(obj, path)


def load_checkpoint(path, q_net, target_net, optimizer, obs_rms):
    obj = torch.load(path, map_location="cpu")
    q_net.load_state_dict(obj["q_net"])
    target_net.load_state_dict(obj["target_net"])
    optimizer.load_state_dict(obj["optimizer"])
    if obs_rms is not None and obj.get("obs_rms") is not None:
        obs_rms.load_state_dict(obj["obs_rms"])
    return obj["state"]


# ============================================================
# Device
# ============================================================
def resolve_device(device_arg: str):
    if device_arg == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    return torch.device(device_arg)


def print_device_info(device):
    print("=" * 72)
    print(f"torch version = {torch.__version__}")
    print(f"cuda available= {torch.cuda.is_available()}")
    print(f"cuda count    = {torch.cuda.device_count()}")
    print(f"device        = {device}")
    if torch.cuda.is_available() and "cuda" in str(device):
        idx = device.index if device.index is not None else 0
        print(f"cuda name     = {torch.cuda.get_device_name(idx)}")
        total_mem = torch.cuda.get_device_properties(idx).total_memory / (1024**3)
        print(f"total memory  = {total_mem:.2f} GB")
    print("=" * 72)


# ============================================================
# Eval
# ============================================================
@torch.no_grad()
def evaluate_policy(q_net, preset: Preset, device, obs_rms: Optional[RunningMeanStd], eval_episodes: int, seed_offset: int = 10000):
    rewards = []
    for ep in range(eval_episodes):
        env = gym.make(preset.env_id)
        obs, _ = env.reset(seed=seed_offset + ep)
        done = False
        truncated = False
        ep_reward = 0.0

        while not (done or truncated):
            obs_np = np.asarray(obs, dtype=np.float32)
            if obs_rms is not None:
                obs_np = obs_rms.normalize(obs_np)
            obs_t = torch.tensor(obs_np, dtype=torch.float32, device=device).unsqueeze(0)
            q = q_net(obs_t)
            action = int(q.argmax(dim=1).item())

            next_obs, raw_reward, done, truncated, _ = env.step(action)
            ep_reward += float(raw_reward)
            obs = next_obs

        rewards.append(ep_reward)
        env.close()

    return float(np.mean(rewards)), float(np.std(rewards))


# ============================================================
# Training step
# ============================================================
def soft_update(target_net, q_net, tau: float):
    for tp, sp in zip(target_net.parameters(), q_net.parameters()):
        tp.data.mul_(1.0 - tau).add_(tau * sp.data)


def train_step(
    q_net,
    target_net,
    optimizer,
    replay: PrioritizedReplayBuffer,
    batch_size: int,
    beta: float,
    gamma_n: float,
    device,
    max_grad_norm: float,
):
    batch = replay.sample(batch_size, beta=beta)

    obs = torch.tensor(batch["obs"], dtype=torch.float32, device=device)
    actions = torch.tensor(batch["actions"], dtype=torch.int64, device=device).unsqueeze(1)
    rewards = torch.tensor(batch["rewards"], dtype=torch.float32, device=device).unsqueeze(1)
    next_obs = torch.tensor(batch["next_obs"], dtype=torch.float32, device=device)
    dones = torch.tensor(batch["dones"], dtype=torch.float32, device=device).unsqueeze(1)
    weights = torch.tensor(batch["weights"], dtype=torch.float32, device=device).unsqueeze(1)

    q_values = q_net(obs).gather(1, actions)

    with torch.no_grad():
        next_actions = q_net(next_obs).argmax(dim=1, keepdim=True)
        next_q = target_net(next_obs).gather(1, next_actions)
        target = rewards + (1.0 - dones) * gamma_n * next_q

    td_error = q_values - target
    loss_per_sample = F.smooth_l1_loss(q_values, target, reduction="none")
    loss = (weights * loss_per_sample).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(q_net.parameters(), max_grad_norm)
    optimizer.step()

    new_priorities = td_error.detach().abs().cpu().numpy().reshape(-1) + 1e-6
    replay.update_priorities(batch["indices"], new_priorities)

    return {
        "loss": float(loss.item()),
        "td_abs": float(td_error.abs().mean().item()),
        "q_mean": float(q_values.mean().item()),
        "target_mean": float(target.mean().item()),
        "grad_norm": float(grad_norm.item()) if hasattr(grad_norm, "item") else float(grad_norm),
    }


# ============================================================
# Train one seed
# ============================================================
def train_one_seed(preset: Preset, seed: int, device, outdir: str, resume: bool):
    env_tmp = gym.make(preset.env_id)
    obs_dim = int(np.prod(env_tmp.observation_space.shape))
    act_dim = int(env_tmp.action_space.n)
    env_tmp.close()

    q_net = DuelingQNetwork(obs_dim, act_dim, preset.hidden_dim).to(device)
    target_net = DuelingQNetwork(obs_dim, act_dim, preset.hidden_dim).to(device)
    target_net.load_state_dict(q_net.state_dict())

    optimizer = torch.optim.Adam(q_net.parameters(), lr=preset.lr, eps=1e-5)
    obs_rms = RunningMeanStd(shape=(obs_dim,)) if preset.use_obs_norm else None
    replay = PrioritizedReplayBuffer(preset.buffer_size, obs_dim, alpha=preset.per_alpha)
    nstep = NStepTransitionAccumulator(preset.n_step, preset.gamma)

    ckpt_dir = os.path.join(outdir, "checkpoints_ddqn")
    ensure_dir(ckpt_dir)
    ckpt_path = os.path.join(ckpt_dir, f"{preset.name}_seed{seed}_latest.pt")

    state = {
        "global_step": 0,
        "episode": 0,
        "best_eval_mean": -1e18,
        "recent_raw_rewards": [],
        "all_raw_rewards": [],
        "first_solve_ep": -1,
        "solved": False,
        "last_eval_mean": 0.0,
        "last_eval_std": 0.0,
    }

    if resume and os.path.exists(ckpt_path):
        print(f"[resume] loading {ckpt_path}")
        state = load_checkpoint(ckpt_path, q_net, target_net, optimizer, obs_rms)
        print(f"[resume] loaded episode={state['episode']} global_step={state['global_step']}")

    global_step = int(state["global_step"])
    episode = int(state["episode"])
    best_eval_mean = float(state["best_eval_mean"])
    recent_raw_rewards = list(state["recent_raw_rewards"])
    all_raw_rewards = list(state["all_raw_rewards"])
    first_solve_ep = int(state["first_solve_ep"])
    solved = bool(state["solved"])
    last_eval_mean = float(state["last_eval_mean"])
    last_eval_std = float(state["last_eval_std"])

    raw_rows = []
    gamma_n = preset.gamma ** preset.n_step

    env = gym.make(preset.env_id)
    obs, _ = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32)
    nstep.reset()

    ep_raw_reward = 0.0
    ep_train_reward = 0.0
    episode_start_step = global_step

    loss_log = deque(maxlen=200)

    while global_step < preset.total_steps and not STOP_REQUESTED:
        progress = global_step / max(1, preset.total_steps - 1)

        epsilon = linear_schedule(
            preset.eps_start,
            preset.eps_end,
            min(1.0, global_step / max(1, preset.eps_decay_steps))
        )
        beta = linear_schedule(
            preset.per_beta_start,
            preset.per_beta_end,
            progress
        )
        current_lr = linear_schedule(preset.lr, preset.min_lr, progress)
        for pg in optimizer.param_groups:
            pg["lr"] = current_lr

        obs_np = np.asarray(obs, dtype=np.float32)
        if obs_rms is not None:
            obs_rms.update(obs_np[None, :])
            obs_in = obs_rms.normalize(obs_np)
        else:
            obs_in = obs_np

        if global_step < preset.warmup_steps or np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                obs_t = torch.tensor(obs_in, dtype=torch.float32, device=device).unsqueeze(0)
                action = int(q_net(obs_t).argmax(dim=1).item())

        next_obs, raw_reward, done, truncated, _ = env.step(action)
        next_obs_np = np.asarray(next_obs, dtype=np.float32)

        train_reward = train_reward_shaping(
            preset.env_id,
            obs_np,
            next_obs_np,
            float(raw_reward),
            bool(done or truncated),
        )

        if obs_rms is not None:
            obs_store = obs_rms.normalize(obs_np)
            next_obs_store = obs_rms.normalize(next_obs_np)
        else:
            obs_store = obs_np
            next_obs_store = next_obs_np

        nstep.append((obs_store, action, train_reward, next_obs_store, float(done or truncated)))

        if nstep.is_ready():
            o, a, r, no, d = nstep.pop_n_step()
            replay.add(o, a, r, no, d)

        obs = next_obs_np
        ep_raw_reward += float(raw_reward)
        ep_train_reward += float(train_reward)
        global_step += 1

        if len(replay) >= preset.batch_size and global_step >= preset.warmup_steps and global_step % preset.train_freq == 0:
            for _ in range(preset.gradient_steps):
                info = train_step(
                    q_net=q_net,
                    target_net=target_net,
                    optimizer=optimizer,
                    replay=replay,
                    batch_size=preset.batch_size,
                    beta=beta,
                    gamma_n=gamma_n,
                    device=device,
                    max_grad_norm=preset.max_grad_norm,
                )
                loss_log.append(info["loss"])

        if global_step % preset.target_update_interval == 0 and len(replay) >= preset.batch_size:
            soft_update(target_net, q_net, preset.tau)

        if done or truncated:
            for o, a, r, no, d in nstep.flush_remaining():
                replay.add(o, a, r, no, d)

            episode += 1
            all_raw_rewards.append(ep_raw_reward)
            recent_raw_rewards.append(ep_raw_reward)
            if len(recent_raw_rewards) > 100:
                recent_raw_rewards = recent_raw_rewards[-100:]

            recent20 = safe_mean(recent_raw_rewards[-20:], 0.0)
            recent100 = safe_mean(recent_raw_rewards[-100:], 0.0)
            mean_loss = safe_mean(loss_log, 0.0)

            raw_rows.append({
                "preset": preset.name,
                "seed": seed,
                "episode": episode,
                "env_id": preset.env_id,
                "global_step": global_step,
                "episode_steps": global_step - episode_start_step,
                "raw_reward": ep_raw_reward,
                "train_reward": ep_train_reward,
                "recent20": recent20,
                "recent100": recent100,
                "epsilon": epsilon,
                "beta": beta,
                "lr": current_lr,
                "buffer_size": len(replay),
                "mean_loss": mean_loss,
            })

            print(
                f"[{preset.name} seed={seed}] "
                f"ep={episode:05d} "
                f"rawR={ep_raw_reward:8.3f} "
                f"trainR={ep_train_reward:8.3f} "
                f"recent20={recent20:8.3f} "
                f"recent100={recent100:8.3f} "
                f"eps={epsilon:.3f} beta={beta:.3f} "
                f"lr={current_lr:.6f} "
                f"buf={len(replay)} loss={mean_loss:.4f} "
                f"global_step={global_step}"
            )

            obs, _ = env.reset(seed=seed + episode)
            obs = np.asarray(obs, dtype=np.float32)
            ep_raw_reward = 0.0
            ep_train_reward = 0.0
            episode_start_step = global_step
            nstep.reset()

            if (not solved) and recent100 >= preset.solve_threshold:
                solved = True
                first_solve_ep = episode
                print(f"[solve] {preset.name} seed={seed} first_solve_ep={first_solve_ep}")

        if global_step % preset.eval_interval_steps == 0 and global_step > 0:
            eval_mean, eval_std = evaluate_policy(
                q_net=q_net,
                preset=preset,
                device=device,
                obs_rms=obs_rms,
                eval_episodes=preset.eval_episodes,
                seed_offset=10000 + seed * 1000 + episode,
            )
            last_eval_mean = eval_mean
            last_eval_std = eval_std
            best_eval_mean = max(best_eval_mean, eval_mean)

            print(
                f"[eval] {preset.name} seed={seed} step={global_step} "
                f"eval_mean={eval_mean:.3f} eval_std={eval_std:.3f}"
            )

            state = {
                "global_step": global_step,
                "episode": episode,
                "best_eval_mean": best_eval_mean,
                "recent_raw_rewards": recent_raw_rewards,
                "all_raw_rewards": all_raw_rewards,
                "first_solve_ep": first_solve_ep,
                "solved": solved,
                "last_eval_mean": last_eval_mean,
                "last_eval_std": last_eval_std,
            }
            save_checkpoint(ckpt_path, q_net, target_net, optimizer, obs_rms, state)

        if STOP_REQUESTED:
            break

        if global_step % 5000 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    env.close()

    state = {
        "global_step": global_step,
        "episode": episode,
        "best_eval_mean": best_eval_mean,
        "recent_raw_rewards": recent_raw_rewards,
        "all_raw_rewards": all_raw_rewards,
        "first_solve_ep": first_solve_ep,
        "solved": solved,
        "last_eval_mean": last_eval_mean,
        "last_eval_std": last_eval_std,
    }
    save_checkpoint(ckpt_path, q_net, target_net, optimizer, obs_rms, state)
    print(f"[ckpt] final saved {ckpt_path}")

    auc_mean = safe_mean(all_raw_rewards, 0.0)
    last100_mean = safe_mean(all_raw_rewards[-100:], 0.0)
    last100_std = float(np.std(all_raw_rewards[-100:])) if len(all_raw_rewards) > 0 else 0.0
    best_reward = max(all_raw_rewards) if len(all_raw_rewards) > 0 else 0.0

    result = {
        "preset": preset.name,
        "seed": seed,
        "env_id": preset.env_id,
        "episodes_completed": episode,
        "auc_mean": auc_mean,
        "last100_mean": last100_mean,
        "last100_std": last100_std,
        "best_reward": best_reward,
        "first_solve_ep": first_solve_ep,
        "eval_mean": last_eval_mean,
        "eval_std": last_eval_std,
        "raw_rows": raw_rows,
    }
    return result


# ============================================================
# IO
# ============================================================
def write_csv(path: str, rows: List[Dict[str, Any]]):
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def write_json(path: str, obj: Any):
    def default(o):
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (np.float32, np.float64)):
            return float(o)
        if isinstance(o, (np.int32, np.int64)):
            return int(o)
        return str(o)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=default)


# ============================================================
# Main
# ============================================================
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--device", type=str, default="cuda:0")
    parser.add_argument("--presets", type=str, default="acrobot_best,mountaincar_best")
    parser.add_argument("--seeds", type=str, default="0,1,2")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--outdir", type=str, default=".")
    return parser.parse_known_args()[0]


def main():
    args = parse_args()
    device = resolve_device(args.device)
    print_device_info(device)

    preset_names = [x.strip() for x in args.presets.split(",") if x.strip()]
    seeds = [int(x.strip()) for x in args.seeds.split(",") if x.strip()]

    invalid = [p for p in preset_names if p not in PRESETS]
    if invalid:
        raise ValueError(f"Unknown presets: {invalid}. Available={list(PRESETS.keys())}")

    ensure_dir(args.outdir)

    all_raw_rows = []
    all_summary_rows = []

    start_time = time.time()

    for preset_name in preset_names:
        preset = PRESETS[preset_name]
        print(f"\n[{preset.name}] env={preset.env_id} total_steps={preset.total_steps}")

        seed_results = []
        for seed in seeds:
            if STOP_REQUESTED:
                break

            set_seed(seed)
            result = train_one_seed(
                preset=preset,
                seed=seed,
                device=device,
                outdir=args.outdir,
                resume=args.resume,
            )
            seed_results.append(result)
            all_raw_rows.extend(result["raw_rows"])

        if len(seed_results) == 0:
            continue

        summary_row = {
            "preset": preset.name,
            "env_id": preset.env_id,
            "n_seeds": len(seed_results),
            "episodes_completed": int(np.mean([r["episodes_completed"] for r in seed_results])),
            "auc_mean": float(np.mean([r["auc_mean"] for r in seed_results])),
            "last100_mean": float(np.mean([r["last100_mean"] for r in seed_results])),
            "last100_std": float(np.mean([r["last100_std"] for r in seed_results])),
            "best_reward": float(np.max([r["best_reward"] for r in seed_results])),
            "first_solve_ep": int(np.mean([r["first_solve_ep"] for r in seed_results])),
            "eval_mean": float(np.mean([r["eval_mean"] for r in seed_results])),
            "eval_std": float(np.mean([r["eval_std"] for r in seed_results])),
        }
        all_summary_rows.append(summary_row)

    raw_csv = os.path.join(args.outdir, "ddqn_acrobot_mountain_raw.csv")
    summary_csv = os.path.join(args.outdir, "ddqn_acrobot_mountain_summary.csv")
    raw_json = os.path.join(args.outdir, "ddqn_acrobot_mountain_raw.json")
    summary_json = os.path.join(args.outdir, "ddqn_acrobot_mountain_summary.json")

    write_csv(raw_csv, all_raw_rows)
    write_csv(summary_csv, all_summary_rows)
    write_json(raw_json, all_raw_rows)
    write_json(summary_json, all_summary_rows)

    elapsed_sec = time.time() - start_time

    print("\n================ FINAL SUMMARY ================")
    print("preset            | env_id         | n_seeds | episodes_completed | auc_mean   | last100_mean | best_reward | first_solve_ep | eval_mean | eval_std")
    print("------------------+----------------+---------+--------------------+------------+--------------+-------------+----------------+-----------+---------")
    for row in all_summary_rows:
        print(
            f"{row['preset']:<18}| "
            f"{row['env_id']:<16}| "
            f"{row['n_seeds']:<8}| "
            f"{row['episodes_completed']:<20}| "
            f"{row['auc_mean']:<10.3f}| "
            f"{row['last100_mean']:<13.3f}| "
            f"{row['best_reward']:<12.3f}| "
            f"{row['first_solve_ep']:<16}| "
            f"{row['eval_mean']:<10.3f}| "
            f"{row['eval_std']:<9.3f}"
        )

    print(f"\nraw csv     : {raw_csv}")
    print(f"summary csv : {summary_csv}")
    print(f"raw json    : {raw_json}")
    print(f"summary json: {summary_json}")
    print(f"elapsed_sec : {elapsed_sec:.2f}")

    print("\n[recommended run]")
    print("python robust_acrobot_mountaincar_ddqn.py --device cuda:0 --presets acrobot_best,mountaincar_best --seeds 0,1,2 --resume")

    print("\n[important note]")
    print("train에서는 shaping reward를 사용하고, eval/summary는 raw reward 기준으로 기록한다.")


if __name__ == "__main__":
    main()

In [1]:
# ============================================================
# check_ablate_outputs.py
# ------------------------------------------------------------
# ablate_runner.py 산출물 검증 코드
# ============================================================

import os
import glob
from collections import defaultdict

try:
    from openpyxl import load_workbook
    OPENPYXL_OK = True
except Exception:
    OPENPYXL_OK = False


# =========================
# 설정
# =========================
XLSX_PATH = "./test_6.xlsx"
CKPT_DIR = "./checkpoints"
LOGDIR = "./runs_ablation"

# ablate_runner.py 기본값
EXTRA_UPDATES_SWEEP = [0, 1, 2, 4, 6]
VARIANTS = ["FULL", "NoSpike", "NoEventUpd", "NoCollapse", "Vanilla"]
SEEDS = [0, 1, 2, 3, 4]

# policy-only 저장 여부 확인용
# save-policy-every를 0보다 크게 줬으면 True로 바꾸는 게 좋음
EXPECT_POLICY_SNAPSHOTS = False


# =========================
# 유틸
# =========================
def ok(msg):
    print(f"[OK] {msg}")


def fail(msg):
    print(f"[FAIL] {msg}")


def warn(msg):
    print(f"[WARN] {msg}")


def exists(path):
    return os.path.exists(path)


def count_rows_in_sheet(ws):
    rows = list(ws.iter_rows(values_only=True))
    return len(rows)


def extract_header(ws):
    rows = list(ws.iter_rows(values_only=True))
    for row in rows:
        if row is None:
            continue
        vals = [x for x in row if x is not None and str(x).strip() != ""]
        if len(vals) >= 2:
            return [str(x).strip() for x in row if x is not None]
    return []


def find_header_row(ws, target_cols):
    rows = list(ws.iter_rows(values_only=True))
    for i, row in enumerate(rows, start=1):
        vals = [("" if x is None else str(x).strip()) for x in row]
        if all(col in vals for col in target_cols):
            return i, vals
    return None, []


def list_pt_files(ckpt_dir):
    return sorted(glob.glob(os.path.join(ckpt_dir, "*.pt")))


def list_meta_files(ckpt_dir):
    return sorted(glob.glob(os.path.join(ckpt_dir, "*_meta.json")))


# =========================
# 체크포인트 이름 파싱
# =========================
def expected_latest_ckpt_names(best_eu=None):
    names = []

    # sweep: FULL only, EU in sweep list
    for eu in EXTRA_UPDATES_SWEEP:
        for seed in SEEDS:
            names.append(f"FULL_EU{eu}_seed{seed}_latest.pt")
            names.append(f"FULL_EU{eu}_seed{seed}_meta.json")

    # ablation: best_eu 하나로 variants 전체
    if best_eu is not None:
        for v in VARIANTS:
            for seed in SEEDS:
                names.append(f"{v}_EU{best_eu}_seed{seed}_latest.pt")
                names.append(f"{v}_EU{best_eu}_seed{seed}_meta.json")

    return names


def parse_best_eu_from_workbook(xlsx_path):
    if not OPENPYXL_OK or not exists(xlsx_path):
        return None

    wb = load_workbook(xlsx_path, data_only=True)
    if "ablation_summary" not in wb.sheetnames:
        return None

    ws = wb["ablation_summary"]
    header_row_idx, header = find_header_row(ws, ["variant", "extra_updates_on_event"])
    if header_row_idx is None:
        return None

    col_map = {name: idx for idx, name in enumerate(header)}
    rows = list(ws.iter_rows(values_only=True))

    values = []
    for row in rows[header_row_idx:]:
        if row is None:
            continue
        variant = row[col_map["variant"]] if col_map["variant"] < len(row) else None
        eu = row[col_map["extra_updates_on_event"]] if col_map["extra_updates_on_event"] < len(row) else None
        if variant is None or eu is None:
            continue
        try:
            values.append(int(float(eu)))
        except Exception:
            pass

    if not values:
        return None

    uniq = sorted(set(values))
    if len(uniq) == 1:
        return uniq[0]

    return uniq[0]


# =========================
# 메인 체크
# =========================
def main():
    print("=============== FILE / FOLDER CHECK ===============")

    if exists(XLSX_PATH):
        ok(f"xlsx exists: {XLSX_PATH}")
    else:
        fail(f"xlsx missing: {XLSX_PATH}")

    if exists(CKPT_DIR):
        ok(f"checkpoint dir exists: {CKPT_DIR}")
    else:
        fail(f"checkpoint dir missing: {CKPT_DIR}")

    if exists(LOGDIR):
        ok(f"logdir exists: {LOGDIR}")
    else:
        warn(f"logdir missing: {LOGDIR} (TensorBoard를 껐으면 정상일 수 있음)")

    print("\n=============== XLSX CHECK ===============")
    if not OPENPYXL_OK:
        fail("openpyxl not installed, cannot inspect xlsx")
        return

    if not exists(XLSX_PATH):
        fail("xlsx file not found, skip workbook checks")
        return

    wb = load_workbook(XLSX_PATH, data_only=True)
    sheetnames = wb.sheetnames
    expected_sheets = ["readme", "sweep_raw", "sweep_summary", "ablation_raw", "ablation_summary"]

    for s in expected_sheets:
        if s in sheetnames:
            ok(f"sheet exists: {s}")
        else:
            fail(f"sheet missing: {s}")

    # 각 시트 비어있는지 확인
    required_sheet_cols = {
        "sweep_raw": ["variant", "seed", "episodes", "extra_updates_on_event", "last100_mean"],
        "sweep_summary": ["group", "variant", "n_seeds", "extra_updates_on_event", "last100_mean"],
        "ablation_raw": ["variant", "seed", "episodes", "extra_updates_on_event", "last100_mean"],
        "ablation_summary": ["variant", "n_seeds", "extra_updates_on_event", "last100_mean"],
    }

    for s, cols in required_sheet_cols.items():
        if s not in wb.sheetnames:
            continue

        ws = wb[s]
        nrows = count_rows_in_sheet(ws)
        if nrows > 0:
            ok(f"{s}: has rows ({nrows})")
        else:
            fail(f"{s}: empty")

        header_row_idx, header = find_header_row(ws, cols[:2])
        if header_row_idx is None:
            fail(f"{s}: header row not found")
            continue

        missing = [c for c in cols if c not in header]
        if missing:
            fail(f"{s}: missing columns {missing}")
        else:
            ok(f"{s}: required columns present")

    print("\n=============== BEST EU DETECTION ===============")
    best_eu = parse_best_eu_from_workbook(XLSX_PATH)
    if best_eu is None:
        warn("best_eu could not be inferred from workbook")
    else:
        ok(f"best_eu inferred from workbook: {best_eu}")

    print("\n=============== CHECKPOINT FILE CHECK ===============")
    pt_files = list_pt_files(CKPT_DIR)
    meta_files = list_meta_files(CKPT_DIR)

    if pt_files:
        ok(f".pt files found: {len(pt_files)}")
    else:
        fail("no .pt files found in checkpoint dir")

    if meta_files:
        ok(f"_meta.json files found: {len(meta_files)}")
    else:
        fail("no _meta.json files found in checkpoint dir")

    # sweep latest 체크
    missing_sweep = []
    for eu in EXTRA_UPDATES_SWEEP:
        for seed in SEEDS:
            pt_name = os.path.join(CKPT_DIR, f"FULL_EU{eu}_seed{seed}_latest.pt")
            meta_name = os.path.join(CKPT_DIR, f"FULL_EU{eu}_seed{seed}_meta.json")
            if not exists(pt_name):
                missing_sweep.append(os.path.basename(pt_name))
            if not exists(meta_name):
                missing_sweep.append(os.path.basename(meta_name))

    if missing_sweep:
        fail(f"sweep checkpoint/meta missing count={len(missing_sweep)}")
        for x in missing_sweep[:15]:
            print("   -", x)
        if len(missing_sweep) > 15:
            print("   ...")
    else:
        ok("all sweep checkpoint/meta files exist")

    # ablation latest 체크
    if best_eu is not None:
        missing_ablation = []
        for v in VARIANTS:
            for seed in SEEDS:
                pt_name = os.path.join(CKPT_DIR, f"{v}_EU{best_eu}_seed{seed}_latest.pt")
                meta_name = os.path.join(CKPT_DIR, f"{v}_EU{best_eu}_seed{seed}_meta.json")
                if not exists(pt_name):
                    missing_ablation.append(os.path.basename(pt_name))
                if not exists(meta_name):
                    missing_ablation.append(os.path.basename(meta_name))

        if missing_ablation:
            fail(f"ablation checkpoint/meta missing count={len(missing_ablation)}")
            for x in missing_ablation[:15]:
                print("   -", x)
            if len(missing_ablation) > 15:
                print("   ...")
        else:
            ok("all ablation checkpoint/meta files exist")
    else:
        warn("skip ablation latest strict check because best_eu is unknown")

    print("\n=============== POLICY SNAPSHOT CHECK ===============")
    policy_files = sorted(glob.glob(os.path.join(CKPT_DIR, "policy_*.pt")))
    if EXPECT_POLICY_SNAPSHOTS:
        if policy_files:
            ok(f"policy snapshot files found: {len(policy_files)}")
        else:
            fail("expected policy snapshots, but none found")
    else:
        if policy_files:
            ok(f"policy snapshot files found: {len(policy_files)}")
        else:
            warn("no policy snapshot files found (save-policy-every=0이면 정상)")

    print("\n=============== TENSORBOARD LOG CHECK ===============")
    if exists(LOGDIR):
        run_dirs = [p for p in glob.glob(os.path.join(LOGDIR, "*")) if os.path.isdir(p)]
        event_files = glob.glob(os.path.join(LOGDIR, "**", "events.out.tfevents.*"), recursive=True)

        if run_dirs:
            ok(f"tensorboard run dirs found: {len(run_dirs)}")
        else:
            warn("no tensorboard run dirs found")

        if event_files:
            ok(f"tensorboard event files found: {len(event_files)}")
        else:
            warn("no tensorboard event files found")
    else:
        warn("logdir not found; tensorboard logging may have been disabled")

    print("\n=============== QUICK SANITY OF WORKBOOK CONTENT ===============")
    if "sweep_summary" in wb.sheetnames:
        ws = wb["sweep_summary"]
        idx, header = find_header_row(ws, ["group", "variant", "extra_updates_on_event", "last100_mean"])
        if idx is not None:
            rows = list(ws.iter_rows(values_only=True))[idx:]
            valid_rows = 0
            for row in rows:
                if row is None:
                    continue
                vals = ["" if x is None else str(x).strip() for x in row]
                if any(v != "" for v in vals):
                    valid_rows += 1
            if valid_rows >= len(EXTRA_UPDATES_SWEEP):
                ok(f"sweep_summary has enough rows: {valid_rows}")
            else:
                warn(f"sweep_summary row count looks small: {valid_rows}")

    if "ablation_summary" in wb.sheetnames:
        ws = wb["ablation_summary"]
        idx, header = find_header_row(ws, ["variant", "last100_mean"])
        if idx is not None:
            rows = list(ws.iter_rows(values_only=True))[idx:]
            valid_rows = 0
            for row in rows:
                if row is None:
                    continue
                vals = ["" if x is None else str(x).strip() for x in row]
                if any(v != "" for v in vals):
                    valid_rows += 1
            if valid_rows >= len(VARIANTS):
                ok(f"ablation_summary has enough rows: {valid_rows}")
            else:
                warn(f"ablation_summary row count looks small: {valid_rows}")

    print("\n=============== FINAL RESULT ===============")
    print("체크포인트(.pt), 메타(json), 엑셀(xlsx), TensorBoard 로그까지 있으면 산출물은 거의 다 나온 상태다.")


if __name__ == "__main__":
    main()

=============== FILE / FOLDER CHECK ===============
[OK] xlsx exists: ./test_6.xlsx
[OK] checkpoint dir exists: ./checkpoints
[WARN] logdir missing: ./runs_ablation (TensorBoard를 껐으면 정상일 수 있음)

=============== XLSX CHECK ===============
[OK] sheet exists: readme
[OK] sheet exists: sweep_raw
[OK] sheet exists: sweep_summary
[OK] sheet exists: ablation_raw
[OK] sheet exists: ablation_summary
[OK] sweep_raw: has rows (28)
[OK] sweep_raw: required columns present
[OK] sweep_summary: has rows (8)
[OK] sweep_summary: required columns present
[OK] ablation_raw: has rows (28)
[OK] ablation_raw: required columns present
[OK] ablation_summary: has rows (8)
[OK] ablation_summary: required columns present

=============== BEST EU DETECTION ===============
[OK] best_eu inferred from workbook: 4

=============== CHECKPOINT FILE CHECK ===============
[OK] .pt files found: 45
[OK] _meta.json files found: 45
[OK] all sweep checkpoint/meta files exist
[OK] all ablation checkpoint/meta files exist

=====

In [ ]:
# ============================================================
# robust_acrobot_mountaincar_event_ddqn_jupyter_safe.py
# ------------------------------------------------------------
# 목표
# - Acrobot-v1 / MountainCar-v0 를 안정적으로 학습
# - Dueling Double DQN + Prioritized Replay + n-step
# - sparse reward 환경 대응
# - train reward shaping / eval raw reward 분리
# - GPU 우선 사용
# - checkpoint / resume / csv / json 저장
#
# + [추가된 핵심 아이디어: event-driven stabilization]
# - episode raw reward 기준 baseline EMA 추적
# - collapse 감지: anchor Q-network에 대한 KL regularization 추가
# - spike 감지: 좋은 episode transition을 spike buffer에 저장 후 extra replay
# - event 발생 시 extra gradient steps 수행
#
# + [Jupyter-safe 저장]
# - atomic temp save -> os.replace
# - legacy torch serialization 사용
# - 저장 실패 시 학습 즉시 종료되지 않도록 보호
#
# Ablation variants
# - FULL         : collapse + spike + event update 모두 사용
# - NoSpike      : spike 재강화 제거
# - NoCollapse   : collapse anchor 제거
# - NoEventUpd   : 이벤트 추가 업데이트 제거
# - Vanilla      : event-driven 장치 전부 제거
# ============================================================

import os
import gc
import csv
import json
import math
import time
import random
import signal
import argparse
import tempfile
from dataclasses import dataclass
from collections import deque
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Global
# ============================================================
STOP_REQUESTED = False


def _signal_handler(signum, frame):
    global STOP_REQUESTED
    STOP_REQUESTED = True
    print(f"\n[signal] received signal={signum}. stopping at next safe point...")


signal.signal(signal.SIGINT, _signal_handler)
signal.signal(signal.SIGTERM, _signal_handler)


# ============================================================
# Event-driven hyperparameters
# ============================================================
BASELINE_BETA = 0.90
EVENT_MIN_INT = 0.04
EPS = 1e-6

AUX_KL_COEF = 0.20
AUX_KL_MAX = 3.0
ANCHOR_EMA_BETA = 0.995

SPIKE_REPLAY_RATIO = 0.50
SPIKE_BUFFER_CAPACITY = 100_000

EXTRA_UPDATES_CAP = 6

SCHEDULE_ON = True
SCHEDULE_START_FRAC = 0.60
KL_SCALE_END = 1.50
SPIKE_SCALE_END = 0.60


# ============================================================
# Variant
# ============================================================
@dataclass
class Variant:
    name: str
    use_collapse_anchor: bool = True
    use_spike_replay: bool = True
    extra_on_collapse: bool = True
    extra_on_spike: bool = True


def get_variants() -> List[Variant]:
    return [
        Variant("FULL",       use_collapse_anchor=True,  use_spike_replay=True,  extra_on_collapse=True,  extra_on_spike=True),
        Variant("NoSpike",    use_collapse_anchor=True,  use_spike_replay=False, extra_on_collapse=True,  extra_on_spike=False),
        Variant("NoCollapse", use_collapse_anchor=False, use_spike_replay=True,  extra_on_collapse=False, extra_on_spike=True),
        Variant("NoEventUpd", use_collapse_anchor=True,  use_spike_replay=True,  extra_on_collapse=False, extra_on_spike=False),
        Variant("Vanilla",    use_collapse_anchor=False, use_spike_replay=False, extra_on_collapse=False, extra_on_spike=False),
    ]


# ============================================================
# Utils
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def safe_mean(xs, default=0.0):
    return float(np.mean(xs)) if len(xs) > 0 else float(default)


def linear_schedule(start: float, end: float, progress: float) -> float:
    progress = float(np.clip(progress, 0.0, 1.0))
    return float(start + (end - start) * progress)


def linear_interp(a: float, b: float, t01: float) -> float:
    t01 = float(max(0.0, min(1.0, t01)))
    return a + (b - a) * t01


def phase_schedule(global_step: int, total_steps: int) -> Dict[str, float]:
    if (not SCHEDULE_ON) or total_steps <= 1:
        return {"kl_scale": 1.0, "spike_scale": 1.0}

    frac = global_step / float(max(1, total_steps - 1))
    if frac <= SCHEDULE_START_FRAC:
        return {"kl_scale": 1.0, "spike_scale": 1.0}

    t = (frac - SCHEDULE_START_FRAC) / (1.0 - SCHEDULE_START_FRAC)
    kl_scale = linear_interp(1.0, KL_SCALE_END, t)
    spike_scale = linear_interp(1.0, SPIKE_SCALE_END, t)
    return {"kl_scale": kl_scale, "spike_scale": spike_scale}


def compute_climb_signals(ep_reward: float, baseline_prev: float):
    denom = abs(baseline_prev) + 1.0
    collapse_raw = max(0.0, baseline_prev - ep_reward)
    spike_raw = max(0.0, ep_reward - baseline_prev)

    collapse_int = collapse_raw / (denom + EPS)
    spike_int = spike_raw / (denom + EPS)

    is_collapse = 1.0 if (ep_reward < baseline_prev and collapse_int >= EVENT_MIN_INT) else 0.0
    is_spike = 1.0 if (ep_reward > baseline_prev and spike_int >= EVENT_MIN_INT) else 0.0

    return is_collapse, is_spike, collapse_int, spike_int


def q_policy_kl(q_now: torch.Tensor, q_anchor: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    log_p_now = F.log_softmax(q_now / temperature, dim=-1)
    p_anchor = F.softmax(q_anchor / temperature, dim=-1)
    return F.kl_div(log_p_now, p_anchor, reduction="batchmean")


def atomic_torch_save(obj, path: str):
    """
    Jupyter/원격 환경에서 torch.save 충돌을 줄이기 위한 안전 저장.
    1) temp 파일에 먼저 저장
    2) 저장 성공 시 os.replace로 원본 교체
    3) legacy serialization 사용
    """
    dirpath = os.path.dirname(path) or "."
    ensure_dir(dirpath)

    fd, tmp_path = tempfile.mkstemp(
        prefix=".tmp_ckpt_",
        suffix=".pt",
        dir=dirpath,
    )
    os.close(fd)

    try:
        torch.save(obj, tmp_path, _use_new_zipfile_serialization=False)
        os.replace(tmp_path, path)
    except Exception:
        try:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
        except Exception:
            pass
        raise


# ============================================================
# Running Mean Std
# ============================================================
class RunningMeanStd:
    def __init__(self, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = 1e-4

    def update(self, x: np.ndarray):
        x = np.asarray(x, dtype=np.float64)
        if x.ndim == 1:
            batch_mean = x
            batch_var = np.zeros_like(x)
            batch_count = 1
        else:
            batch_mean = x.mean(axis=0)
            batch_var = x.var(axis=0)
            batch_count = x.shape[0]
        self._update_from_moments(batch_mean, batch_var, batch_count)

    def _update_from_moments(self, batch_mean, batch_var, batch_count):
        delta = batch_mean - self.mean
        total_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / total_count

        m_a = self.var * self.count
        m_b = batch_var * batch_count
        m2 = m_a + m_b + (delta ** 2) * self.count * batch_count / total_count
        new_var = m2 / total_count

        self.mean = new_mean
        self.var = np.maximum(new_var, 1e-8)
        self.count = total_count

    def normalize(self, x: np.ndarray, clip: float = 10.0) -> np.ndarray:
        x = (x - self.mean) / np.sqrt(self.var + 1e-8)
        return np.clip(x, -clip, clip)

    def state_dict(self):
        return {"mean": self.mean, "var": self.var, "count": self.count}

    def load_state_dict(self, d):
        self.mean = d["mean"]
        self.var = d["var"]
        self.count = d["count"]


# ============================================================
# Presets
# ============================================================
@dataclass
class Preset:
    name: str
    env_id: str
    total_steps: int
    warmup_steps: int
    buffer_size: int
    batch_size: int
    hidden_dim: int
    lr: float
    min_lr: float
    gamma: float
    n_step: int
    target_update_interval: int
    tau: float
    eps_start: float
    eps_end: float
    eps_decay_steps: int
    per_alpha: float
    per_beta_start: float
    per_beta_end: float
    train_freq: int
    gradient_steps: int
    max_grad_norm: float
    eval_interval_steps: int
    eval_episodes: int
    solve_threshold: float
    solve_window: int
    use_obs_norm: bool


PRESETS = {
    "acrobot_best": Preset(
        name="acrobot_best",
        env_id="Acrobot-v1",
        total_steps=450_000,
        warmup_steps=10_000,
        buffer_size=250_000,
        batch_size=256,
        hidden_dim=256,
        lr=2.5e-4,
        min_lr=8.0e-5,
        gamma=0.99,
        n_step=5,
        target_update_interval=1,
        tau=0.01,
        eps_start=1.0,
        eps_end=0.03,
        eps_decay_steps=180_000,
        per_alpha=0.6,
        per_beta_start=0.4,
        per_beta_end=1.0,
        train_freq=1,
        gradient_steps=1,
        max_grad_norm=10.0,
        eval_interval_steps=10_000,
        eval_episodes=10,
        solve_threshold=-90.0,
        solve_window=100,
        use_obs_norm=True,
    ),
    "mountaincar_best": Preset(
        name="mountaincar_best",
        env_id="MountainCar-v0",
        total_steps=350_000,
        warmup_steps=8_000,
        buffer_size=200_000,
        batch_size=256,
        hidden_dim=256,
        lr=2.0e-4,
        min_lr=6.0e-5,
        gamma=0.997,
        n_step=3,
        target_update_interval=1,
        tau=0.01,
        eps_start=1.0,
        eps_end=0.02,
        eps_decay_steps=140_000,
        per_alpha=0.6,
        per_beta_start=0.4,
        per_beta_end=1.0,
        train_freq=1,
        gradient_steps=1,
        max_grad_norm=10.0,
        eval_interval_steps=8_000,
        eval_episodes=10,
        solve_threshold=-110.0,
        solve_window=100,
        use_obs_norm=True,
    ),
}


# ============================================================
# Reward shaping (train only)
# ============================================================
def train_reward_shaping(env_id: str, obs: np.ndarray, next_obs: np.ndarray, raw_reward: float, done: bool) -> float:
    shaped = float(raw_reward)

    if env_id == "MountainCar-v0":
        pos, vel = next_obs
        prev_pos, _prev_vel = obs

        progress = 6.0 * (pos - prev_pos)
        speed_bonus = 1.5 * abs(vel)
        uphill_bonus = 0.5 * max(0.0, pos + 0.5)

        terminal_bonus = 0.0
        if done and pos >= 0.5:
            terminal_bonus = 120.0

        shaped = raw_reward + progress + speed_bonus + uphill_bonus + terminal_bonus

    elif env_id == "Acrobot-v1":
        c1, s1, c2, s2, d1, d2 = next_obs
        pc1, ps1, pc2, ps2, pd1, pd2 = obs

        cur_height_like = -(c1 + c1 * c2 - s1 * s2)
        prev_height_like = -(pc1 + pc1 * pc2 - ps1 * ps2)

        height_progress = 3.5 * (cur_height_like - prev_height_like)
        height_bonus = 0.5 * cur_height_like
        vel_penalty = -0.002 * (abs(d1) + abs(d2))

        terminal_bonus = 0.0
        if done:
            terminal_bonus = 25.0

        shaped = raw_reward + height_progress + height_bonus + vel_penalty + terminal_bonus

    return float(shaped)


# ============================================================
# Dueling Q Network
# ============================================================
class DuelingQNetwork(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        self.adv_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=math.sqrt(2))
                nn.init.constant_(m.bias, 0.0)
        nn.init.orthogonal_(self.value_stream[-1].weight, gain=1.0)
        nn.init.orthogonal_(self.adv_stream[-1].weight, gain=0.01)

    def forward(self, x):
        z = self.feature(x)
        v = self.value_stream(z)
        a = self.adv_stream(z)
        q = v + (a - a.mean(dim=-1, keepdim=True))
        return q


# ============================================================
# Prioritized Replay with n-step
# ============================================================
class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, obs_dim: int, alpha: float = 0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.pos = 0
        self.size = 0

        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros((capacity,), dtype=np.int64)
        self.rewards = np.zeros((capacity,), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.dones = np.zeros((capacity,), dtype=np.float32)
        self.priorities = np.zeros((capacity,), dtype=np.float32)

    def add(self, obs, action, reward, next_obs, done, priority: Optional[float] = None):
        self.obs[self.pos] = obs
        self.actions[self.pos] = action
        self.rewards[self.pos] = reward
        self.next_obs[self.pos] = next_obs
        self.dones[self.pos] = float(done)

        if priority is None:
            max_prio = self.priorities.max() if self.size > 0 else 1.0
            self.priorities[self.pos] = max_prio
        else:
            self.priorities[self.pos] = max(float(priority), 1e-6)

        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int, beta: float):
        if self.size == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.size]

        probs = prios ** self.alpha
        probs_sum = probs.sum()
        if probs_sum <= 0:
            probs = np.ones_like(probs) / len(probs)
        else:
            probs /= probs_sum

        indices = np.random.choice(self.size, batch_size, p=probs)
        weights = (self.size * probs[indices]) ** (-beta)
        weights /= weights.max()

        batch = dict(
            obs=self.obs[indices],
            actions=self.actions[indices],
            rewards=self.rewards[indices],
            next_obs=self.next_obs[indices],
            dones=self.dones[indices],
            indices=indices,
            weights=weights.astype(np.float32),
        )
        return batch

    def update_priorities(self, indices, priorities):
        priorities = np.asarray(priorities, dtype=np.float32)
        self.priorities[indices] = np.maximum(priorities, 1e-6)

    def __len__(self):
        return self.size


class NStepTransitionAccumulator:
    def __init__(self, n_step: int, gamma: float):
        self.n_step = n_step
        self.gamma = gamma
        self.buffer = deque(maxlen=n_step)

    def reset(self):
        self.buffer.clear()

    def append(self, transition):
        self.buffer.append(transition)

    def is_ready(self):
        return len(self.buffer) >= self.n_step

    def pop_n_step(self):
        reward = 0.0
        next_obs = None
        done = False

        obs, action, _, _, _ = self.buffer[0]

        for i, (_, _, r, nobs, d) in enumerate(self.buffer):
            reward += (self.gamma ** i) * r
            next_obs = nobs
            done = d
            if d:
                break

        self.buffer.popleft()
        return obs, action, reward, next_obs, done

    def flush_remaining(self):
        items = []
        while len(self.buffer) > 0:
            reward = 0.0
            next_obs = None
            done = False
            obs, action, _, _, _ = self.buffer[0]

            for i, (_, _, r, nobs, d) in enumerate(self.buffer):
                reward += (self.gamma ** i) * r
                next_obs = nobs
                done = d
                if d:
                    break

            self.buffer.popleft()
            items.append((obs, action, reward, next_obs, done))
        return items


# ============================================================
# Checkpoint
# ============================================================
def save_checkpoint(path, q_net, target_net, anchor_net, optimizer, obs_rms, state):
    obj = {
        "q_net": q_net.state_dict(),
        "target_net": target_net.state_dict(),
        "anchor_net": anchor_net.state_dict() if anchor_net is not None else None,
        "optimizer": optimizer.state_dict(),
        "obs_rms": obs_rms.state_dict() if obs_rms is not None else None,
        "state": state,
    }
    atomic_torch_save(obj, path)


def load_checkpoint(path, q_net, target_net, anchor_net, optimizer, obs_rms):
    obj = torch.load(path, map_location="cpu")
    q_net.load_state_dict(obj["q_net"])
    target_net.load_state_dict(obj["target_net"])
    if anchor_net is not None and obj.get("anchor_net") is not None:
        anchor_net.load_state_dict(obj["anchor_net"])
    optimizer.load_state_dict(obj["optimizer"])
    if obs_rms is not None and obj.get("obs_rms") is not None:
        obs_rms.load_state_dict(obj["obs_rms"])
    return obj["state"]


# ============================================================
# Device
# ============================================================
def resolve_device(device_arg: str):
    if device_arg == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    return torch.device(device_arg)


def print_device_info(device):
    print("=" * 72)
    print(f"torch version = {torch.__version__}")
    print(f"cuda available= {torch.cuda.is_available()}")
    print(f"cuda count    = {torch.cuda.device_count()}")
    print(f"device        = {device}")
    if torch.cuda.is_available() and "cuda" in str(device):
        idx = device.index if device.index is not None else 0
        print(f"cuda name     = {torch.cuda.get_device_name(idx)}")
        total_mem = torch.cuda.get_device_properties(idx).total_memory / (1024**3)
        print(f"total memory  = {total_mem:.2f} GB")
    print("=" * 72)


# ============================================================
# Eval
# ============================================================
@torch.no_grad()
def evaluate_policy(q_net, preset: Preset, device, obs_rms: Optional[RunningMeanStd], eval_episodes: int, seed_offset: int = 10000):
    rewards = []
    for ep in range(eval_episodes):
        env = gym.make(preset.env_id)
        obs, _ = env.reset(seed=seed_offset + ep)
        done = False
        truncated = False
        ep_reward = 0.0

        while not (done or truncated):
            obs_np = np.asarray(obs, dtype=np.float32)
            if obs_rms is not None:
                obs_np = obs_rms.normalize(obs_np)
            obs_t = torch.tensor(obs_np, dtype=torch.float32, device=device).unsqueeze(0)
            q = q_net(obs_t)
            action = int(q.argmax(dim=1).item())

            next_obs, raw_reward, done, truncated, _ = env.step(action)
            ep_reward += float(raw_reward)
            obs = next_obs

        rewards.append(ep_reward)
        env.close()

    return float(np.mean(rewards)), float(np.std(rewards))


# ============================================================
# Training helpers
# ============================================================
def soft_update(target_net, q_net, tau: float):
    for tp, sp in zip(target_net.parameters(), q_net.parameters()):
        tp.data.mul_(1.0 - tau).add_(tau * sp.data)


@torch.no_grad()
def ema_update_anchor(anchor_net, q_net, beta: float):
    for ap, qp in zip(anchor_net.parameters(), q_net.parameters()):
        ap.data.mul_(beta).add_(qp.data, alpha=(1.0 - beta))


def build_tensors_from_batch(batch, device):
    obs = torch.tensor(batch["obs"], dtype=torch.float32, device=device)
    actions = torch.tensor(batch["actions"], dtype=torch.int64, device=device).unsqueeze(1)
    rewards = torch.tensor(batch["rewards"], dtype=torch.float32, device=device).unsqueeze(1)
    next_obs = torch.tensor(batch["next_obs"], dtype=torch.float32, device=device)
    dones = torch.tensor(batch["dones"], dtype=torch.float32, device=device).unsqueeze(1)
    weights = torch.tensor(batch["weights"], dtype=torch.float32, device=device).unsqueeze(1)
    return obs, actions, rewards, next_obs, dones, weights


def merge_batches(base_batch, spike_batch, spike_ratio: float):
    if spike_batch is None:
        return base_batch

    k1 = len(base_batch["actions"])
    k2 = len(spike_batch["actions"])

    merged = {}
    for key in ["obs", "actions", "rewards", "next_obs", "dones", "weights"]:
        merged[key] = np.concatenate([base_batch[key], spike_batch[key]], axis=0)

    merged["indices"] = base_batch["indices"]
    merged["base_size"] = k1
    merged["spike_size"] = k2
    merged["spike_ratio"] = spike_ratio
    return merged


def train_step_event(
    q_net,
    target_net,
    anchor_net,
    optimizer,
    replay: PrioritizedReplayBuffer,
    spike_replay: Optional[PrioritizedReplayBuffer],
    batch_size: int,
    beta: float,
    gamma_n: float,
    device,
    max_grad_norm: float,
    collapse_kl_weight: float = 0.0,
    use_spike_batch: bool = False,
    spike_batch_ratio: float = SPIKE_REPLAY_RATIO,
):
    base_bs = batch_size
    spike_bs = 0

    if use_spike_batch and spike_replay is not None and len(spike_replay) >= max(16, batch_size // 4):
        spike_bs = int(round(batch_size * spike_batch_ratio))
        spike_bs = max(1, min(batch_size - 1, spike_bs))
        base_bs = batch_size - spike_bs

    base_batch = replay.sample(base_bs, beta=beta)

    spike_batch = None
    if spike_bs > 0:
        spike_batch = spike_replay.sample(spike_bs, beta=beta)

    batch = merge_batches(base_batch, spike_batch, spike_batch_ratio) if spike_batch is not None else base_batch
    obs, actions, rewards, next_obs, dones, weights = build_tensors_from_batch(batch, device)

    q_values_all = q_net(obs)
    q_values = q_values_all.gather(1, actions)

    with torch.no_grad():
        next_actions = q_net(next_obs).argmax(dim=1, keepdim=True)
        next_q = target_net(next_obs).gather(1, next_actions)
        target = rewards + (1.0 - dones) * gamma_n * next_q

    td_error = q_values - target
    loss_per_sample = F.smooth_l1_loss(q_values, target, reduction="none")
    td_loss = (weights * loss_per_sample).mean()

    kl_loss = torch.tensor(0.0, device=device)
    if collapse_kl_weight > 0.0 and anchor_net is not None:
        with torch.no_grad():
            q_anchor = anchor_net(obs)
        kl = q_policy_kl(q_values_all, q_anchor)
        kl_loss = collapse_kl_weight * kl

    loss = td_loss + kl_loss

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(q_net.parameters(), max_grad_norm)
    optimizer.step()

    base_size = batch.get("base_size", len(base_batch["actions"]))
    td_error_base = td_error[:base_size]
    new_priorities = td_error_base.detach().abs().cpu().numpy().reshape(-1) + 1e-6
    replay.update_priorities(base_batch["indices"], new_priorities)

    return {
        "loss": float(loss.item()),
        "td_loss": float(td_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "td_abs": float(td_error.abs().mean().item()),
        "q_mean": float(q_values.mean().item()),
        "target_mean": float(target.mean().item()),
        "grad_norm": float(grad_norm.item()) if hasattr(grad_norm, "item") else float(grad_norm),
        "used_spike_batch": int(spike_bs > 0),
    }


# ============================================================
# Train one seed
# ============================================================
def train_one_seed(
    preset: Preset,
    variant: Variant,
    extra_updates_on_event: int,
    seed: int,
    device,
    outdir: str,
    resume: bool,
):
    env_tmp = gym.make(preset.env_id)
    obs_dim = int(np.prod(env_tmp.observation_space.shape))
    act_dim = int(env_tmp.action_space.n)
    env_tmp.close()

    q_net = DuelingQNetwork(obs_dim, act_dim, preset.hidden_dim).to(device)
    target_net = DuelingQNetwork(obs_dim, act_dim, preset.hidden_dim).to(device)
    target_net.load_state_dict(q_net.state_dict())

    anchor_net = DuelingQNetwork(obs_dim, act_dim, preset.hidden_dim).to(device)
    anchor_net.load_state_dict(q_net.state_dict())
    anchor_net.eval()

    optimizer = torch.optim.Adam(q_net.parameters(), lr=preset.lr, eps=1e-5)
    obs_rms = RunningMeanStd(shape=(obs_dim,)) if preset.use_obs_norm else None

    replay = PrioritizedReplayBuffer(preset.buffer_size, obs_dim, alpha=preset.per_alpha)
    spike_replay = PrioritizedReplayBuffer(SPIKE_BUFFER_CAPACITY, obs_dim, alpha=preset.per_alpha)

    nstep = NStepTransitionAccumulator(preset.n_step, preset.gamma)

    ckpt_dir = os.path.join(outdir, "checkpoints_event_ddqn")
    ensure_dir(ckpt_dir)
    ckpt_path = os.path.join(
        ckpt_dir,
        f"{preset.name}_{variant.name}_EU{extra_updates_on_event}_seed{seed}_latest.pt"
    )

    state = {
        "global_step": 0,
        "episode": 0,
        "best_eval_mean": -1e18,
        "recent_raw_rewards": [],
        "all_raw_rewards": [],
        "first_solve_ep": -1,
        "solved": False,
        "last_eval_mean": 0.0,
        "last_eval_std": 0.0,
        "baseline": 0.0,
        "collapse_count": 0,
        "spike_count": 0,
    }

    if resume and os.path.exists(ckpt_path):
        print(f"[resume] loading {ckpt_path}")
        state = load_checkpoint(ckpt_path, q_net, target_net, anchor_net, optimizer, obs_rms)
        print(f"[resume] loaded episode={state['episode']} global_step={state['global_step']}")

    global_step = int(state["global_step"])
    episode = int(state["episode"])
    best_eval_mean = float(state["best_eval_mean"])
    recent_raw_rewards = list(state["recent_raw_rewards"])
    all_raw_rewards = list(state["all_raw_rewards"])
    first_solve_ep = int(state["first_solve_ep"])
    solved = bool(state["solved"])
    last_eval_mean = float(state["last_eval_mean"])
    last_eval_std = float(state["last_eval_std"])

    baseline = float(state.get("baseline", 0.0))
    collapse_count = int(state.get("collapse_count", 0))
    spike_count = int(state.get("spike_count", 0))

    raw_rows = []
    gamma_n = preset.gamma ** preset.n_step

    env = gym.make(preset.env_id)
    obs, _ = env.reset(seed=seed)
    obs = np.asarray(obs, dtype=np.float32)
    nstep.reset()

    ep_raw_reward = 0.0
    ep_train_reward = 0.0
    episode_start_step = global_step

    loss_log = deque(maxlen=200)
    kl_log = deque(maxlen=200)
    current_episode_transitions = []

    while global_step < preset.total_steps and not STOP_REQUESTED:
        progress = global_step / max(1, preset.total_steps - 1)

        epsilon = linear_schedule(
            preset.eps_start,
            preset.eps_end,
            min(1.0, global_step / max(1, preset.eps_decay_steps))
        )
        beta = linear_schedule(
            preset.per_beta_start,
            preset.per_beta_end,
            progress
        )
        current_lr = linear_schedule(preset.lr, preset.min_lr, progress)
        for pg in optimizer.param_groups:
            pg["lr"] = current_lr

        sched = phase_schedule(global_step, preset.total_steps)
        kl_scale = sched["kl_scale"]
        spike_scale = sched["spike_scale"]

        obs_np = np.asarray(obs, dtype=np.float32)
        if obs_rms is not None:
            obs_rms.update(obs_np[None, :])
            obs_in = obs_rms.normalize(obs_np)
        else:
            obs_in = obs_np

        if global_step < preset.warmup_steps or np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                obs_t = torch.tensor(obs_in, dtype=torch.float32, device=device).unsqueeze(0)
                action = int(q_net(obs_t).argmax(dim=1).item())

        next_obs, raw_reward, done, truncated, _ = env.step(action)
        next_obs_np = np.asarray(next_obs, dtype=np.float32)

        train_reward = train_reward_shaping(
            preset.env_id,
            obs_np,
            next_obs_np,
            float(raw_reward),
            bool(done or truncated),
        )

        if obs_rms is not None:
            obs_store = obs_rms.normalize(obs_np)
            next_obs_store = obs_rms.normalize(next_obs_np)
        else:
            obs_store = obs_np
            next_obs_store = next_obs_np

        nstep.append((obs_store, action, train_reward, next_obs_store, float(done or truncated)))

        if nstep.is_ready():
            o, a, r, no, d = nstep.pop_n_step()
            replay.add(o, a, r, no, d)
            current_episode_transitions.append((o, a, r, no, d))

        obs = next_obs_np
        ep_raw_reward += float(raw_reward)
        ep_train_reward += float(train_reward)
        global_step += 1

        if len(replay) >= preset.batch_size and global_step >= preset.warmup_steps and global_step % preset.train_freq == 0:
            for _ in range(preset.gradient_steps):
                info = train_step_event(
                    q_net=q_net,
                    target_net=target_net,
                    anchor_net=anchor_net if variant.use_collapse_anchor else None,
                    optimizer=optimizer,
                    replay=replay,
                    spike_replay=spike_replay if variant.use_spike_replay else None,
                    batch_size=preset.batch_size,
                    beta=beta,
                    gamma_n=gamma_n,
                    device=device,
                    max_grad_norm=preset.max_grad_norm,
                    collapse_kl_weight=0.0,
                    use_spike_batch=False,
                )
                loss_log.append(info["loss"])
                kl_log.append(info["kl_loss"])

        if global_step % preset.target_update_interval == 0 and len(replay) >= preset.batch_size:
            soft_update(target_net, q_net, preset.tau)

        if done or truncated:
            for o, a, r, no, d in nstep.flush_remaining():
                replay.add(o, a, r, no, d)
                current_episode_transitions.append((o, a, r, no, d))

            episode += 1
            all_raw_rewards.append(ep_raw_reward)
            recent_raw_rewards.append(ep_raw_reward)
            if len(recent_raw_rewards) > 100:
                recent_raw_rewards = recent_raw_rewards[-100:]

            baseline_prev = float(baseline)
            is_collapse, is_spike, collapse_int, spike_int = compute_climb_signals(ep_raw_reward, baseline_prev)
            baseline = BASELINE_BETA * baseline + (1.0 - BASELINE_BETA) * float(ep_raw_reward)

            if is_collapse:
                collapse_count += 1
            if is_spike:
                spike_count += 1

            if is_spike and variant.use_spike_replay:
                prio_boost = 1.0 + 5.0 * spike_int * spike_scale
                for (o, a, r, no, d) in current_episode_transitions:
                    spike_replay.add(o, a, r, no, d, priority=prio_boost)

            collapse_kl_weight = 0.0
            if variant.use_collapse_anchor and is_collapse:
                collapse_kl_weight = min(AUX_KL_MAX, AUX_KL_COEF * (1.0 + 5.0 * collapse_int))
                collapse_kl_weight *= kl_scale

            extra_updates = 0
            if is_collapse and variant.extra_on_collapse:
                extra_updates = max(extra_updates, extra_updates_on_event)
            if is_spike and variant.extra_on_spike:
                extra_updates = max(extra_updates, extra_updates_on_event)
            extra_updates = int(min(EXTRA_UPDATES_CAP, extra_updates))

            if len(replay) >= preset.batch_size:
                for _ in range(extra_updates):
                    use_spike_batch = bool(
                        is_spike and variant.use_spike_replay and len(spike_replay) >= max(16, preset.batch_size // 4)
                    )
                    info = train_step_event(
                        q_net=q_net,
                        target_net=target_net,
                        anchor_net=anchor_net if variant.use_collapse_anchor else None,
                        optimizer=optimizer,
                        replay=replay,
                        spike_replay=spike_replay if variant.use_spike_replay else None,
                        batch_size=preset.batch_size,
                        beta=beta,
                        gamma_n=gamma_n,
                        device=device,
                        max_grad_norm=preset.max_grad_norm,
                        collapse_kl_weight=collapse_kl_weight if is_collapse else 0.0,
                        use_spike_batch=use_spike_batch,
                    )
                    loss_log.append(info["loss"])
                    kl_log.append(info["kl_loss"])

            if variant.use_collapse_anchor:
                ema_update_anchor(anchor_net, q_net, ANCHOR_EMA_BETA)
                anchor_net.eval()

            recent20 = safe_mean(recent_raw_rewards[-20:], 0.0)
            recent100 = safe_mean(recent_raw_rewards[-100:], 0.0)
            mean_loss = safe_mean(loss_log, 0.0)
            mean_kl = safe_mean(kl_log, 0.0)

            raw_rows.append({
                "preset": preset.name,
                "variant": variant.name,
                "seed": seed,
                "episode": episode,
                "env_id": preset.env_id,
                "global_step": global_step,
                "episode_steps": global_step - episode_start_step,
                "raw_reward": ep_raw_reward,
                "train_reward": ep_train_reward,
                "recent20": recent20,
                "recent100": recent100,
                "epsilon": epsilon,
                "beta": beta,
                "lr": current_lr,
                "buffer_size": len(replay),
                "spike_buffer_size": len(spike_replay),
                "mean_loss": mean_loss,
                "mean_kl": mean_kl,
                "baseline": baseline,
                "is_collapse": is_collapse,
                "is_spike": is_spike,
                "collapse_int": collapse_int,
                "spike_int": spike_int,
                "extra_updates": extra_updates,
            })

            tag = "COLLAPSE" if is_collapse else ("SPIKE" if is_spike else "normal")
            print(
                f"[{preset.name} {variant.name} seed={seed}] "
                f"ep={episode:05d} "
                f"rawR={ep_raw_reward:8.3f} "
                f"trainR={ep_train_reward:8.3f} "
                f"recent100={recent100:8.3f} "
                f"ema={baseline:8.3f} "
                f"{tag} "
                f"extra={extra_updates} "
                f"eps={epsilon:.3f} beta={beta:.3f} "
                f"lr={current_lr:.6f} "
                f"buf={len(replay)} spike_buf={len(spike_replay)} "
                f"loss={mean_loss:.4f} kl={mean_kl:.4f} "
                f"global_step={global_step}"
            )

            obs, _ = env.reset(seed=seed + episode)
            obs = np.asarray(obs, dtype=np.float32)
            ep_raw_reward = 0.0
            ep_train_reward = 0.0
            episode_start_step = global_step
            nstep.reset()
            current_episode_transitions = []

            if (not solved) and recent100 >= preset.solve_threshold:
                solved = True
                first_solve_ep = episode
                print(f"[solve] {preset.name} {variant.name} seed={seed} first_solve_ep={first_solve_ep}")

        if global_step % preset.eval_interval_steps == 0 and global_step > 0:
            eval_mean, eval_std = evaluate_policy(
                q_net=q_net,
                preset=preset,
                device=device,
                obs_rms=obs_rms,
                eval_episodes=preset.eval_episodes,
                seed_offset=10000 + seed * 1000 + episode,
            )
            last_eval_mean = eval_mean
            last_eval_std = eval_std
            best_eval_mean = max(best_eval_mean, eval_mean)

            print(
                f"[eval] {preset.name} {variant.name} seed={seed} step={global_step} "
                f"eval_mean={eval_mean:.3f} eval_std={eval_std:.3f}"
            )

            state = {
                "global_step": global_step,
                "episode": episode,
                "best_eval_mean": best_eval_mean,
                "recent_raw_rewards": recent_raw_rewards,
                "all_raw_rewards": all_raw_rewards,
                "first_solve_ep": first_solve_ep,
                "solved": solved,
                "last_eval_mean": last_eval_mean,
                "last_eval_std": last_eval_std,
                "baseline": baseline,
                "collapse_count": collapse_count,
                "spike_count": spike_count,
            }

            try:
                save_checkpoint(ckpt_path, q_net, target_net, anchor_net, optimizer, obs_rms, state)
                print(f"[ckpt] saved {ckpt_path}")
            except Exception as e:
                print(f"[warn] checkpoint save failed at step={global_step}: {repr(e)}")

        if STOP_REQUESTED:
            break

        if global_step % 5000 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    env.close()

    state = {
        "global_step": global_step,
        "episode": episode,
        "best_eval_mean": best_eval_mean,
        "recent_raw_rewards": recent_raw_rewards,
        "all_raw_rewards": all_raw_rewards,
        "first_solve_ep": first_solve_ep,
        "solved": solved,
        "last_eval_mean": last_eval_mean,
        "last_eval_std": last_eval_std,
        "baseline": baseline,
        "collapse_count": collapse_count,
        "spike_count": spike_count,
    }

    try:
        save_checkpoint(ckpt_path, q_net, target_net, anchor_net, optimizer, obs_rms, state)
        print(f"[ckpt] final saved {ckpt_path}")
    except Exception as e:
        print(f"[warn] final checkpoint save failed: {repr(e)}")

    auc_mean = safe_mean(all_raw_rewards, 0.0)
    last100_mean = safe_mean(all_raw_rewards[-100:], 0.0)
    last100_std = float(np.std(all_raw_rewards[-100:])) if len(all_raw_rewards) > 0 else 0.0
    best_reward = max(all_raw_rewards) if len(all_raw_rewards) > 0 else 0.0

    result = {
        "preset": preset.name,
        "variant": variant.name,
        "seed": seed,
        "env_id": preset.env_id,
        "episodes_completed": episode,
        "auc_mean": auc_mean,
        "last100_mean": last100_mean,
        "last100_std": last100_std,
        "best_reward": best_reward,
        "first_solve_ep": first_solve_ep,
        "eval_mean": last_eval_mean,
        "eval_std": last_eval_std,
        "collapse_count": collapse_count,
        "spike_count": spike_count,
        "extra_updates_on_event": extra_updates_on_event,
        "raw_rows": raw_rows,
    }
    return result


# ============================================================
# IO
# ============================================================
def write_csv(path: str, rows: List[Dict[str, Any]]):
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)


def write_json(path: str, obj: Any):
    def default(o):
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (np.float32, np.float64)):
            return float(o)
        if isinstance(o, (np.int32, np.int64)):
            return int(o)
        return str(o)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=default)


# ============================================================
# Main
# ============================================================
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--device", type=str, default="cuda:0")
    parser.add_argument("--presets", type=str, default="acrobot_best,mountaincar_best")
    parser.add_argument("--variants", type=str, default="FULL,NoSpike,NoCollapse,NoEventUpd,Vanilla")
    parser.add_argument("--extra-updates", type=int, default=2)
    parser.add_argument("--seeds", type=str, default="0,1,2")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--outdir", type=str, default="./event_ddqn_runs")
    args, _unknown = parser.parse_known_args()
    return args


def main():
    args = parse_args()
    device = resolve_device(args.device)
    print_device_info(device)

    preset_names = [x.strip() for x in args.presets.split(",") if x.strip()]
    seeds = [int(x.strip()) for x in args.seeds.split(",") if x.strip()]
    variant_names = [x.strip() for x in args.variants.split(",") if x.strip()]

    invalid = [p for p in preset_names if p not in PRESETS]
    if invalid:
        raise ValueError(f"Unknown presets: {invalid}. Available={list(PRESETS.keys())}")

    all_variants = {v.name: v for v in get_variants()}
    invalid_variants = [v for v in variant_names if v not in all_variants]
    if invalid_variants:
        raise ValueError(f"Unknown variants: {invalid_variants}. Available={list(all_variants.keys())}")

    ensure_dir(args.outdir)

    all_raw_rows = []
    all_summary_rows = []

    start_time = time.time()

    for preset_name in preset_names:
        preset = PRESETS[preset_name]
        print(f"\n[{preset.name}] env={preset.env_id} total_steps={preset.total_steps}")

        for variant_name in variant_names:
            variant = all_variants[variant_name]
            print(f"\n--- Variant: {variant.name} | extra_updates={args.extra_updates} ---")

            seed_results = []
            for seed in seeds:
                if STOP_REQUESTED:
                    break

                set_seed(seed)
                result = train_one_seed(
                    preset=preset,
                    variant=variant,
                    extra_updates_on_event=int(args.extra_updates),
                    seed=seed,
                    device=device,
                    outdir=args.outdir,
                    resume=args.resume,
                )
                seed_results.append(result)
                all_raw_rows.extend(result["raw_rows"])

            if len(seed_results) == 0:
                continue

            summary_row = {
                "preset": preset.name,
                "variant": variant.name,
                "env_id": preset.env_id,
                "n_seeds": len(seed_results),
                "episodes_completed": int(np.mean([r["episodes_completed"] for r in seed_results])),
                "auc_mean": float(np.mean([r["auc_mean"] for r in seed_results])),
                "last100_mean": float(np.mean([r["last100_mean"] for r in seed_results])),
                "last100_std": float(np.mean([r["last100_std"] for r in seed_results])),
                "best_reward": float(np.max([r["best_reward"] for r in seed_results])),
                "first_solve_ep": int(np.mean([r["first_solve_ep"] for r in seed_results])),
                "eval_mean": float(np.mean([r["eval_mean"] for r in seed_results])),
                "eval_std": float(np.mean([r["eval_std"] for r in seed_results])),
                "collapse_count": float(np.mean([r["collapse_count"] for r in seed_results])),
                "spike_count": float(np.mean([r["spike_count"] for r in seed_results])),
                "extra_updates_on_event": int(args.extra_updates),
            }
            all_summary_rows.append(summary_row)

    raw_csv = os.path.join(args.outdir, "event_ddqn_raw.csv")
    summary_csv = os.path.join(args.outdir, "event_ddqn_summary.csv")
    raw_json = os.path.join(args.outdir, "event_ddqn_raw.json")
    summary_json = os.path.join(args.outdir, "event_ddqn_summary.json")

    write_csv(raw_csv, all_raw_rows)
    write_csv(summary_csv, all_summary_rows)
    write_json(raw_json, all_raw_rows)
    write_json(summary_json, all_summary_rows)

    elapsed_sec = time.time() - start_time

    print("\n================ FINAL SUMMARY ================")
    print("preset            | variant       | env_id         | n_seeds | last100_mean | eval_mean | first_solve_ep | collapse_count | spike_count | EU")
    print("------------------+---------------+----------------+---------+--------------+-----------+----------------+----------------+-------------+----")
    for row in all_summary_rows:
        print(
            f"{row['preset']:<18}| "
            f"{row['variant']:<14}| "
            f"{row['env_id']:<16}| "
            f"{row['n_seeds']:<8}| "
            f"{row['last100_mean']:<13.3f}| "
            f"{row['eval_mean']:<10.3f}| "
            f"{row['first_solve_ep']:<16}| "
            f"{row['collapse_count']:<15.2f}| "
            f"{row['spike_count']:<12.2f}| "
            f"{row['extra_updates_on_event']:<3}"
        )

    print(f"\nraw csv     : {raw_csv}")
    print(f"summary csv : {summary_csv}")
    print(f"raw json    : {raw_json}")
    print(f"summary json: {summary_json}")
    print(f"elapsed_sec : {elapsed_sec:.2f}")

    print("\n[recommended run - terminal]")
    print(
        "python robust_acrobot_mountaincar_event_ddqn_jupyter_safe.py "
        "--device cuda:0 "
        "--presets acrobot_best,mountaincar_best "
        "--variants FULL,NoSpike,NoCollapse,NoEventUpd,Vanilla "
        "--extra-updates 2 "
        "--seeds 0,1,2 "
        "--resume "
        "--outdir ./event_ddqn_runs"
    )

    print("\n[important note]")
    print("train에서는 shaping reward를 사용하고, eval/summary/event baseline은 raw reward 기준으로 기록한다.")


if __name__ == "__main__":
    main()

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch version = 2.10.0+cu128
cuda available= True
cuda count    = 1
device        = cuda:0
cuda name     = NVIDIA L40S
total memory  = 44.52 GB

[acrobot_best] env=Acrobot-v1 total_steps=450000

--- Variant: FULL | extra_updates=2 ---
[acrobot_best FULL seed=0] ep=00001 rawR=-500.000 trainR=-895.789 recent100=-500.000 ema= -50.000 COLLAPSE extra=2 eps=0.997 beta=0.401 lr=0.000250 buf=500 spike_buf=0 loss=7.2108 kl=0.0001 global_step=500
[acrobot_best FULL seed=0] ep=00002 rawR=-500.000 trainR=-943.923 recent100=-500.000 ema= -95.000 COLLAPSE extra=2 eps=0.995 beta=0.401 lr=0.000250 buf=1000 spike_buf=0 loss=5.8836 kl=0.0006 global_step=1000
[acrobot_best FULL seed=0] ep=00003 rawR=-500.000 trainR=-826.494 recent100=-500.000 ema=-135.500 COLLAPSE extra=2 eps=0.992 beta=0.402 lr=0.000249 buf=1500 spike_buf=0 loss=5.3587 kl=0.0009 global_step=1500
[acrobot_best FULL seed=0] ep=00004 rawR=-500.000 trainR=-928.565 recent100=-500.000 ema=-171.950 COLLAPSE extra=2 eps=0.989 beta=0.403 lr=0.00